# V1 EfficientNetB4 GAP+GMP + CBLoss

Generated from `efficientnetb4-ablation-cbloss-only.ipynb` using the requirements in `GDE_Net_prompt_versions.md`.

This notebook keeps the original data pipeline, augmentation, 3-run training loop, callbacks, fine-tuning strategy, TTA inference, and metric reporting. Only the model head/loss variant is changed for ablation.


In [ ]:
# ============================================================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# ============================================================================
# This notebook is self-contained and does not depend on external notebooks.
# ============================================================================

import os
import textwrap
import csv
import time
import random
import shutil
import glob
import gc
import json
from collections import defaultdict
from types import SimpleNamespace

# --- Scientific Computing ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Deep Learning Framework ---
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Layer, Input, Concatenate, GlobalMaxPooling2D,
    Reshape, Multiply, Add, Activation,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, CSVLogger, Callback,
)
from tensorflow.keras.regularizers import l2

# --- Scikit-learn Metrics & Utilities ---
from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score, cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ============================================================================
# GPU CONFIGURATION + MIXED PRECISION
# ============================================================================
print("=" * 60)
print("  ENVIRONMENT SETUP - V1 GAP+GMP")
print("=" * 60)
print(f"  TensorFlow version : {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"  GPU(s) detected    : {len(gpus)} - {[g.name for g in gpus]}")
else:
    print("  [WARN]  No GPU detected - training will be slow!")

tf.keras.mixed_precision.set_global_policy("mixed_float16")
print(f"  Mixed Precision    : mixed_float16")
print(f"  XLA JIT            : Enabled")
print("=" * 60)


# ============================================================================
# PAPER-READY FIGURE AND REPORT HELPERS
# ============================================================================
# These helpers keep labels, titles, and saved reports consistent across all
# runs. They use readable class names for figures while preserving the raw
# folder names for data loading and indexing.
EXPERIMENT_DISPLAY_NAME = "EfficientNetB4 GAP/GMP with class-balanced focal loss"

CLASS_NAME_MAP = {
    "Fully_Peeled_Garlic": "Fully peeled garlic",
    "Partially_Peeled_Garlic": "Partially peeled garlic",
    "Spoiled_Garlic": "Spoiled garlic",
}


def format_class_label(label):
    """Return a reader-facing class label for paper figures and reports."""
    raw = str(label)
    if raw in CLASS_NAME_MAP:
        return CLASS_NAME_MAP[raw]
    text = raw.replace("_", " ").strip()
    if not text:
        return raw
    return text[:1].upper() + text[1:].lower()


def plain_class_names(labels):
    """Return unwrapped display labels for tables, legends, and text reports."""
    return [format_class_label(label) for label in labels]


def display_class_names(labels, width=18):
    """Return wrapped display labels so class ticks fit compact figures."""
    return ["\n".join(textwrap.wrap(format_class_label(label), width=max(1, width)))
            for label in labels]


def experiment_display_name():
    """Return the model/strategy label used in scientific figure titles."""
    for key in ("STRATEGY_LABEL", "MODEL_LABEL", "EXPERIMENT_DISPLAY_NAME"):
        value = globals().get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return "Garlic classification model"


def format_confusion_matrix_axes(ax):
    """Apply consistent axis labels and tick layout to confusion matrices."""
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.tick_params(axis="x", labelrotation=35)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")
    for tick in ax.get_yticklabels():
        tick.set_rotation(0)
        tick.set_va("center")


In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION & HYPERPARAMETERS
# ============================================================================
# --- Experiment Identification ---
STRATEGY_KEY   = "v1_efficientnetb4_gap_gmp_cbloss"
STRATEGY_LABEL = "V1 EfficientNetB4 GAP+GMP + CBLoss"
MODEL_VARIANT  = "v1_efficientnetb4_gap_gmp_cbloss"

# --- Data Paths ---
DATA_DIR        = "/kaggle/input/datasets/usser12aa/dataset/dataset_split_0803"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
RESULTS_CSV_NAME = "V1_EfficientNetB4_GAP_GMP_CBLoss_results.csv"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model Architecture ---
INPUT_SHAPE     = (380, 380, 3)       # EfficientNetB4 standard
BATCH_SIZE      = 32
EPOCHS          = 50
LR              = 8e-5
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]
DROPOUT_RATE    = 0.25
PATIENCE        = 10

# --- Optimization and generalization controls ---
WEIGHT_DECAY    = 1e-5
LABEL_SMOOTHING = 0.03
TTA_ROUNDS      = 4
TTA_INFER_BATCH = 8

# --- GDE-Net ablation switches ---
FEAT_DIM             = 256
SE_REDUCTION         = 8       # kept for API compatibility with the baseline builder
EVIDENCE_DIM         = 64
DEFECT_TOPK_RATIO    = 0.10
DIVERSITY_LAMBDA     = 0.03
USE_GAP_GMP          = True
USE_COVERAGE_BRANCH  = False
USE_DEFECT_BRANCH    = False
USE_DIVERSITY_LOSS   = False
USE_AUXILIARY_OUTPUTS = False  # kept False to preserve the baseline tf.data/training loop

# --- Loss ---
LOSS_TYPE       = "cbloss"
FOCAL_GAMMA     = 2.0
LOSS_NAME       = "Class-Balanced Focal Loss"
LOSS_DESCRIPTION = f"Class-Balanced Focal Loss (gamma={FOCAL_GAMMA}, beta=0.9999)"

ARCHITECTURE_SUMMARY = [
    "Backbone: EfficientNetB4 (unfreeze [3, 4, 5, 6, 7])",
    "Features: final EfficientNetB4 semantic map -> GAP + GMP",
    "Classifier: BN -> Dense(256) -> Dropout -> Softmax",
    "Loss: Class-Balanced Focal Loss",
]

# --- Reproducibility ---
N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]
AUTOTUNE     = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)
all_runs_results = []

# --- Print Summary ---
print("=" * 60)
print("  EXPERIMENT CONFIGURATION")
print("=" * 60)
print(f"  Strategy    : {STRATEGY_LABEL}")
print(f"  Dataset     : {DATA_DIR.split('/')[-1]}")
print(f"  Input Shape : {INPUT_SHAPE}")
print(f"  Batch Size  : {BATCH_SIZE}")
print(f"  Epochs      : {EPOCHS} (patience={PATIENCE})")
print(f"  LR          : {LR} (CosineDecay -> 1e-6)")
print(f"  Optimizer   : AdamW (wd={WEIGHT_DECAY}, clipnorm=1.0)")
print(f"  Label smooth: {LABEL_SMOOTHING}")
print(f"  TTA rounds  : {TTA_ROUNDS}")
print(f"  TTA infer bs: {TTA_INFER_BATCH}")
print(f"  Unfreeze    : blocks {UNFREEZE_BLOCKS}")
print(f"  Runs        : {N_RUNS} x seeds {RANDOM_SEEDS}")
print("-" * 60)
print("  [Ablation] Architecture:")
for item in ARCHITECTURE_SUMMARY:
    print(f"             - {item}")
print(f"  [Loss]    {LOSS_DESCRIPTION}")
print("=" * 60)


In [ ]:
# ============================================================================
# CELL 3: GDE-NET ABLATION ARCHITECTURE (Keras 3 compatible)
# ============================================================================
# All versions keep the same EfficientNetB4 backbone and training protocol.
# The switches in Cell 2 control whether this becomes V0, V1, V2, V3, V4,
# V5, V6, or V7.
# ============================================================================


@tf.keras.utils.register_keras_serializable(package="GDE")
class CastToFloat32(Layer):
    """Cast tensors to float32 before logits/softmax under mixed precision."""
    def call(self, x):
        return tf.cast(x, tf.float32)


@tf.keras.utils.register_keras_serializable(package="GDE")
class TopKPooling2D(Layer):
    """Return mean(top-k spatial activations) and max activation for a map."""
    def __init__(self, k_ratio=0.10, **kwargs):
        super().__init__(**kwargs)
        self.k_ratio = float(k_ratio)
        self.k = None

    def build(self, input_shape):
        h, w = input_shape[1], input_shape[2]
        if h is None or w is None:
            raise ValueError("TopKPooling2D requires static spatial dimensions.")
        self.k = max(1, int(round(float(h * w) * self.k_ratio)))
        super().build(input_shape)

    def call(self, x):
        x = tf.cast(x, tf.float32)
        flat = tf.reshape(x, [tf.shape(x)[0], -1])
        top_values = tf.math.top_k(flat, k=self.k, sorted=False).values
        topk_mean = tf.reduce_mean(top_values, axis=-1, keepdims=True)
        topk_max = tf.reduce_max(top_values, axis=-1, keepdims=True)
        return [topk_mean, topk_max]

    def compute_output_shape(self, input_shape):
        return [(input_shape[0], 1), (input_shape[0], 1)]

    def get_config(self):
        cfg = super().get_config()
        cfg["k_ratio"] = self.k_ratio
        return cfg


@tf.keras.utils.register_keras_serializable(package="GDE")
class DiversityRegularizer(Layer):
    """Add lambda * mean(coverage_map * defect_map) as a model loss."""
    def __init__(self, lambda_div=0.03, **kwargs):
        super().__init__(**kwargs)
        self.lambda_div = float(lambda_div)

    def call(self, inputs):
        coverage_map, defect_map = inputs
        coverage_map = tf.cast(coverage_map, tf.float32)
        defect_map = tf.cast(defect_map, tf.float32)
        self.add_loss(self.lambda_div * tf.reduce_mean(coverage_map * defect_map))
        return coverage_map

    def get_config(self):
        cfg = super().get_config()
        cfg["lambda_div"] = self.lambda_div
        return cfg


@tf.keras.utils.register_keras_serializable(package="GDE")
class GatedLogitFusion(Layer):
    """Fuse global logits with one or two evidence-logit branches."""
    def __init__(self, mode="dual", **kwargs):
        super().__init__(**kwargs)
        self.mode = mode

    def call(self, inputs):
        if self.mode == "dual":
            global_logits, coverage_logits, defect_logits, gate = inputs
            global_logits = tf.cast(global_logits, tf.float32)
            coverage_logits = tf.cast(coverage_logits, tf.float32)
            defect_logits = tf.cast(defect_logits, tf.float32)
            gate = tf.cast(gate, tf.float32)
            alpha_cov = gate[:, 0:1]
            alpha_def = gate[:, 1:2]
            return global_logits + alpha_cov * coverage_logits + alpha_def * defect_logits

        global_logits, evidence_logits, alpha = inputs
        return (
            tf.cast(global_logits, tf.float32)
            + tf.cast(alpha, tf.float32) * tf.cast(evidence_logits, tf.float32)
        )

    def get_config(self):
        cfg = super().get_config()
        cfg["mode"] = self.mode
        return cfg


def _global_feature_head(feature_map, feat_dim, dropout_rate):
    """Global classifier feature used by all variants."""
    gap = GlobalAveragePooling2D(name="gap_global")(feature_map)
    if USE_GAP_GMP:
        gmp = GlobalMaxPooling2D(name="gmp_global")(feature_map)
        x = Concatenate(name="global_gap_gmp")([gap, gmp])
    else:
        x = gap

    x = BatchNormalization(name="head_bn")(x)
    x = Dense(feat_dim, activation="relu", kernel_regularizer=l2(1e-4), name="head_dense")(x)
    x = Dropout(dropout_rate, name="head_dropout")(x)
    return CastToFloat32(name="head_feature_f32")(x)


def _coverage_branch(shared_feature, num_classes):
    coverage_map = Conv2D(1, 1, padding="same", activation="sigmoid", name="coverage_map")(shared_feature)
    cov_mean = GlobalAveragePooling2D(name="coverage_mean")(coverage_map)
    cov_max = GlobalMaxPooling2D(name="coverage_max")(coverage_map)
    cov_stats = Concatenate(name="coverage_stats")([cov_mean, cov_max])
    cov_feature = Dense(EVIDENCE_DIM, activation="swish", name="coverage_feature")(cov_stats)
    cov_feature = CastToFloat32(name="coverage_feature_f32")(cov_feature)
    cov_logits = Dense(num_classes, dtype="float32", name="coverage_logits")(cov_feature)
    return coverage_map, cov_feature, cov_logits


def _defect_branch(shared_feature, num_classes):
    defect_map = Conv2D(1, 1, padding="same", activation="sigmoid", name="defect_map")(shared_feature)
    topk_mean, topk_max = TopKPooling2D(k_ratio=DEFECT_TOPK_RATIO, name="defect_topk_pool")(defect_map)
    def_stats = Concatenate(name="defect_stats")([topk_mean, topk_max])
    def_feature = Dense(EVIDENCE_DIM, activation="swish", name="defect_feature")(def_stats)
    def_feature = CastToFloat32(name="defect_feature_f32")(def_feature)
    def_logits = Dense(num_classes, dtype="float32", name="defect_logits")(def_feature)
    return defect_map, def_feature, def_logits


def build_mscaf_classifier(input_shape, num_classes, feat_dim=256,
                           se_reduction=8, dropout_rate=0.4):
    """Build the selected EfficientNetB4/GDE-Net ablation classifier."""
    backbone_base = EfficientNetB4(weights="imagenet", include_top=False, input_shape=input_shape)
    inputs = Input(shape=input_shape, name="input_image")
    semantic_map = backbone_base(inputs)

    if USE_COVERAGE_BRANCH or USE_DEFECT_BRANCH:
        shared_feature = Conv2D(256, 1, padding="same", activation="swish", name="shared_reduce_conv")(semantic_map)
        shared_feature = BatchNormalization(name="shared_reduce_bn")(shared_feature)
    else:
        shared_feature = semantic_map

    global_feature = _global_feature_head(shared_feature, feat_dim, dropout_rate)
    global_logits = Dense(num_classes, dtype="float32", name="global_logits")(global_feature)

    coverage_map = coverage_feature = coverage_logits = None
    defect_map = defect_feature = defect_logits = None

    if USE_COVERAGE_BRANCH:
        coverage_map, coverage_feature, coverage_logits = _coverage_branch(shared_feature, num_classes)

    if USE_DEFECT_BRANCH:
        defect_map, defect_feature, defect_logits = _defect_branch(shared_feature, num_classes)

    if USE_DIVERSITY_LOSS and USE_COVERAGE_BRANCH and USE_DEFECT_BRANCH:
        coverage_map = DiversityRegularizer(
            lambda_div=DIVERSITY_LAMBDA,
            name="diversity_regularizer",
        )([coverage_map, defect_map])
        # Recompute coverage statistics so the add_loss layer stays connected
        # to the final model graph.
        cov_mean = GlobalAveragePooling2D(name="coverage_mean_div")(coverage_map)
        cov_max = GlobalMaxPooling2D(name="coverage_max_div")(coverage_map)
        cov_stats = Concatenate(name="coverage_stats_div")([cov_mean, cov_max])
        coverage_feature = Dense(EVIDENCE_DIM, activation="swish", name="coverage_feature_div")(cov_stats)
        coverage_feature = CastToFloat32(name="coverage_feature_div_f32")(coverage_feature)
        coverage_logits = Dense(num_classes, dtype="float32", name="coverage_logits_div")(coverage_feature)

    if USE_COVERAGE_BRANCH and USE_DEFECT_BRANCH:
        gate_input = Concatenate(name="evidence_gate_input")([
            global_feature, coverage_feature, defect_feature,
        ])
        gate = Dense(2, activation="sigmoid", dtype="float32", name="evidence_gate")(gate_input)
        final_logits = GatedLogitFusion(mode="dual", name="evidence_logit_fusion")([
            global_logits, coverage_logits, defect_logits, gate,
        ])
    elif USE_COVERAGE_BRANCH:
        alpha = Dense(1, activation="sigmoid", dtype="float32", name="coverage_gate")(global_feature)
        final_logits = GatedLogitFusion(mode="single", name="coverage_logit_fusion")([
            global_logits, coverage_logits, alpha,
        ])
    elif USE_DEFECT_BRANCH:
        alpha = Dense(1, activation="sigmoid", dtype="float32", name="defect_gate")(global_feature)
        final_logits = GatedLogitFusion(mode="single", name="defect_logit_fusion")([
            global_logits, defect_logits, alpha,
        ])
    else:
        final_logits = global_logits

    outputs = tf.keras.layers.Softmax(name="final_softmax", dtype="float32")(final_logits)
    model = Model(inputs=inputs, outputs=outputs, name=MODEL_VARIANT)
    return model, backbone_base


print("[OK] GDE-Net ablation architecture ready")
print(f"   - GAP+GMP: {USE_GAP_GMP}")
print(f"   - Coverage branch: {USE_COVERAGE_BRANCH}")
print(f"   - Defect branch: {USE_DEFECT_BRANCH}")
print(f"   - Diversity loss: {USE_DIVERSITY_LOSS} (lambda={DIVERSITY_LAMBDA})")
print(f"   - Auxiliary outputs: {USE_AUXILIARY_OUTPUTS} (disabled to keep baseline pipeline)")


In [ ]:
# ============================================================================
# CELL 4: LOSS FUNCTION - Class-Balanced Focal Loss
# ============================================================================
# This version uses Class-Balanced Focal Loss as the main objective.
# Supervised contrastive learning can be added later as an optional auxiliary phase.
#
# Rationale for excluding joint SupCon training in this notebook:
#   - custom train_step with Keras 3 can complicate callback-monitored loss tracking
#   - SupCon generally benefits from larger batches to ensure enough positive pairs
#   - V1 GAP+GMP + CB Focal provides a clean and strong baseline for ablation-first evaluation
# ============================================================================


class ClassBalancedFocalLoss(tf.keras.losses.Loss):
    """Class-Balanced Focal Loss.
    
    CB weights (Cui et al., CVPR 2019): effective number of samples
    Focal (Lin et al., ICCV 2017): down-weight easy examples
    
    Combined: handles both class imbalance AND easy/hard example imbalance.
    """
    def __init__(self, samples_per_class, num_classes, gamma=2.0, beta=0.9999, label_smoothing=0.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.beta = beta
        self.num_classes = num_classes
        self.label_smoothing = float(label_smoothing)
        self._samples_per_class = list(samples_per_class)
        
        # Compute CB weights: w_i = (1-beta) / (1-beta^n_i), normalized
        n = np.array(samples_per_class, dtype=np.float32)
        eff_num = 1.0 - np.power(beta, n)
        weights = (1.0 - beta) / eff_num
        weights = weights / weights.sum() * num_classes
        self.cb_weights = tf.constant(weights, dtype=tf.float32)
        print(f"  CB weights: {dict(zip(range(num_classes), weights.round(4)))}")

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_true_hard = y_true

        # Label smoothing to improve calibration/generalization
        if self.label_smoothing > 0.0:
            y_true = y_true * (1.0 - self.label_smoothing) + (self.label_smoothing / self.num_classes)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Per-sample class weight (use hard labels)
        sample_w = tf.reduce_sum(y_true_hard * self.cb_weights, axis=-1)
        
        # Focal modulation: (1 - p_t)^gamma
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal = tf.pow(1.0 - pt, self.gamma)
        
        # Cross-entropy
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        
        return tf.reduce_mean(sample_w * focal * ce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'samples_per_class': self._samples_per_class,
                    'num_classes': self.num_classes,
                    'gamma': self.gamma, 'beta': self.beta,
                    'label_smoothing': self.label_smoothing})
        return cfg


print("[OK] ClassBalancedFocalLoss ready (CB + Focal)")
print("   - SupCon is intentionally excluded in this version")
print("   - Focus: isolate V1 GAP+GMP impact with a robust and reproducible objective")


In [ ]:
# ============================================================================
# CELL 5: DATA PIPELINE
# ============================================================================
# Returns ONE-HOT labels (for standard Keras training, not integer labels)
# ============================================================================

efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

# Moderate augmentation: improve generalization while preserving class semantics.
_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomTranslation(0.10, 0.10),
    tf.keras.layers.RandomBrightness(factor=0.15),
    tf.keras.layers.RandomContrast(factor=0.15),
], name='augmentation')


def _collect_samples(split_dir, class_to_idx):
    """Collect all image paths + labels from a split directory."""
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        if not os.path.isdir(d):
            continue
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None):
    """Create train/val/test tf.data pipelines with one-hot labels for multi-class training."""
    class_names = sorted([d for d in os.listdir(os.path.join(data_dir, 'train'))
                          if os.path.isdir(os.path.join(data_dir, 'train', d))])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes = len(class_names)
    h, w = input_shape[:2]

    def load_and_preprocess(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [h, w])
        img = tf.cast(img, tf.float32)
        img = efficientnet_preprocess(img)
        # Return ONE-HOT label (standard Keras)
        return img, tf.one_hot(label, depth=num_classes)

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False):
        sdir = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training).prefetch(AUTOTUNE)
        return ds, len(paths), fns, labels

    train_ds, n_train, _, train_lbl = _make_split('train', training=True)
    val_ds, n_val, _, _ = _make_split('val', training=False)
    test_ds, n_test, test_fnames, test_lbl = _make_split('test', training=False)

    # Samples per class for loss/reporting
    samples_per_class = [train_lbl.count(i) for i in range(num_classes)]
    
    meta = SimpleNamespace(
        class_names=class_names, num_classes=num_classes,
        test_filenames=test_fnames, test_classes=np.array(test_lbl),
        n_train=n_train, n_val=n_val, n_test=n_test,
        samples_per_class=samples_per_class,
    )
    print(f"  Data: train={n_train} val={n_val} test={n_test}")
    print(f"  Classes: {class_names}")
    print(f"  Samples/class (train): {samples_per_class}")
    return train_ds, val_ds, test_ds, meta


print("[OK] Data pipeline defined (one-hot labels for standard Keras training).")


In [ ]:
# ============================================================================
# CELL 6: MODEL BUILDER (Functional API)
# ============================================================================
# Uses standard model.compile(loss=...) so Keras can track loss consistently.
# The dataset and 3-run loop are unchanged from the baseline notebook.
# ============================================================================


def apply_freeze_strategy(base_model, unfreeze_blocks):
    """Freeze backbone except specified blocks. Keep BN frozen."""
    base_model.trainable = False
    for layer in base_model.layers:
        for block_num in unfreeze_blocks:
            if layer.name.startswith(f"block{block_num}"):
                if not isinstance(layer, tf.keras.layers.BatchNormalization):
                    layer.trainable = True
                break
    trainable = sum(1 for l in base_model.layers if l.trainable)
    total = len(base_model.layers)
    print(f"  Backbone: {trainable}/{total} layers trainable")


def build_and_compile_model(num_classes, samples_per_class, steps_per_epoch):
    """Build, freeze, and compile the selected ablation model."""
    model, backbone_base = build_mscaf_classifier(
        input_shape=INPUT_SHAPE,
        num_classes=num_classes,
        feat_dim=FEAT_DIM,
        se_reduction=SE_REDUCTION,
        dropout_rate=DROPOUT_RATE,
    )

    apply_freeze_strategy(backbone_base, UNFREEZE_BLOCKS)

    if LOSS_TYPE.lower() == "ce":
        loss_fn = tf.keras.losses.CategoricalCrossentropy(
            label_smoothing=LABEL_SMOOTHING,
            name="categorical_crossentropy",
        )
    else:
        loss_fn = ClassBalancedFocalLoss(
            samples_per_class=samples_per_class,
            num_classes=num_classes,
            gamma=FOCAL_GAMMA,
            beta=0.9999,
            label_smoothing=LABEL_SMOOTHING,
        )

    total_steps = steps_per_epoch * EPOCHS
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LR,
        decay_steps=total_steps,
        alpha=1e-6,
    )
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=WEIGHT_DECAY,
        clipnorm=1.0,
    )

    auc_metric = tf.keras.metrics.AUC(
        name="auc_ovr",
        curve="ROC",
        multi_label=True,
        num_labels=num_classes,
    )

    model.compile(
        optimizer=optimizer,
        loss=loss_fn,
        metrics=["accuracy", auc_metric],
    )

    print(f"  Model params: {model.count_params():,}")
    if len(model.losses) > 0:
        print(f"  Extra model losses: {len(model.losses)} (e.g. diversity regularization)")
    return model


print("[OK] Model builder ready")
print("   - Keras loss tracking: enabled")
print("   - EarlyStopping on val_auc_ovr")
print("   - CosineDecay LR schedule")


In [ ]:
# ============================================================================
# CELL 7: MULTI-RUN TRAINING LOOP
# ============================================================================
# This cell executes N independent runs and persists run-level artifacts so
# downstream thesis/paper analysis can be reproduced without re-training.
# ============================================================================
def predict_with_tta(model, ds, tta_rounds=0, tta_infer_batch=8):
    """Average predictions from original and random TTA forward passes.

    To avoid GPU OOM in multi-run experiments, inference is executed using a
    smaller micro-batch and dtype aligned with mixed precision policy.
    """
    infer_ds = ds.unbatch().batch(tta_infer_batch).prefetch(AUTOTUNE)

    probs = model.predict(infer_ds, verbose=0)
    if tta_rounds <= 0:
        return probs

    probs_sum = probs.astype(np.float32)
    compute_dtype = tf.as_dtype(tf.keras.mixed_precision.global_policy().compute_dtype)
    out_spec = tf.TensorSpec(shape=tf.TensorShape(INPUT_SHAPE), dtype=compute_dtype)

    def _augment_one(img):
        img = tf.cast(img, compute_dtype)
        aug = _augmentation(img, training=True)
        return tf.cast(aug, compute_dtype)

    for _ in range(tta_rounds):
        aug_probs = []
        for batch_x, _ in infer_ds:
            batch_x = tf.cast(batch_x, compute_dtype)
            batch_aug = tf.map_fn(_augment_one, batch_x, fn_output_signature=out_spec)
            try:
                probs_batch = model.predict_on_batch(batch_aug)
            except tf.errors.ResourceExhaustedError:
                micro = max(1, tta_infer_batch // 2)
                parts = []
                n = int(batch_aug.shape[0]) if batch_aug.shape[0] is not None else int(tf.shape(batch_aug)[0].numpy())
                for s in range(0, n, micro):
                    parts.append(model.predict_on_batch(batch_aug[s:s + micro]))
                probs_batch = np.concatenate(parts, axis=0)
            aug_probs.append(np.asarray(probs_batch, dtype=np.float32))
        probs_sum += np.concatenate(aug_probs, axis=0)

    return probs_sum / float(tta_rounds + 1)


for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print()
    print("=" * 70)
    print(f" RUN {run_idx + 1}/{N_RUNS}  seed={seed}  |  {STRATEGY_LABEL}")
    print("=" * 70)

    # Clear residual state before each run.
    tf.keras.backend.clear_session()
    plt.close('all')
    gc.collect()

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx + 1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    # --- Data ---
    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed)
    steps_per_epoch = max(meta.n_train // BATCH_SIZE, 1)

    # --- Build Model ---
    model = build_and_compile_model(
        num_classes=meta.num_classes,
        samples_per_class=meta.samples_per_class,
        steps_per_epoch=steps_per_epoch,
    )

    if run_idx == 0:
        print()
        print("  Model architecture:")
        for item in ARCHITECTURE_SUMMARY:
            print(f"    {item}")
        print(f"    Loss: {LOSS_DESCRIPTION}")
        print(f"    LR: CosineDecay({LR} -> 1e-6)")

    callbacks = [
        EarlyStopping(
            monitor='val_auc_ovr', mode='max', patience=PATIENCE,
            restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv')),
        ModelCheckpoint(
            os.path.join(RESULT_DIR, 'best_model.keras'),
            save_best_only=True, monitor='val_auc_ovr', mode='max', verbose=1),
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    # --- Evaluate on test ---
    pred_probs = predict_with_tta(
        model,
        test_ds,
        tta_rounds=TTA_ROUNDS,
        tta_infer_batch=TTA_INFER_BATCH,
    )
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes

    report = classification_report(
        y_true_run, y_pred_run,
        target_names=meta.class_names, output_dict=True, digits=4)

    test_acc = float(np.mean(y_pred_run == y_true_run))
    bal_acc = float(balanced_accuracy_score(y_true_run, y_pred_run))
    kappa = float(cohen_kappa_score(y_true_run, y_pred_run))
    mcc = float(matthews_corrcoef(y_true_run, y_pred_run))

    y_true_onehot = label_binarize(y_true_run, classes=np.arange(meta.num_classes))
    try:
        test_auc_macro = float(roc_auc_score(
            y_true_onehot, pred_probs,
            average='macro', multi_class='ovr'))
        test_auc_weighted = float(roc_auc_score(
            y_true_onehot, pred_probs,
            average='weighted', multi_class='ovr'))
    except ValueError:
        test_auc_macro = np.nan
        test_auc_weighted = np.nan

    # --- Persist run artifacts required for thesis/paper analysis ---
    np.save(os.path.join(RESULT_DIR, 'y_true.npy'), y_true_run)
    np.save(os.path.join(RESULT_DIR, 'y_pred.npy'), y_pred_run)
    np.save(os.path.join(RESULT_DIR, 'pred_probs.npy'), pred_probs)
    np.save(os.path.join(RESULT_DIR, 'history.npy'), history.history, allow_pickle=True)
    with open(os.path.join(RESULT_DIR, 'test_filenames.json'), 'w', encoding='utf-8') as f:
        json.dump(meta.test_filenames, f, ensure_ascii=False)

    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w', encoding='utf-8') as f:
        f.write(classification_report(
            y_true_run, y_pred_run,
            target_names=meta.class_names, digits=4))

    # --- Confusion matrix ---
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        xticklabels=display_class_names(meta.class_names),
        yticklabels=display_class_names(meta.class_names),
        cmap='Blues',
        ax=ax,
    )
    ax.set_title(f'{experiment_display_name()}: test-set confusion matrix (run {run_idx + 1}, test accuracy={test_acc:.4f})', fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.show()
    plt.close(fig)

    # --- ROC curves (OvR) ---
    fig, ax = plt.subplots(figsize=(8, 6))
    for ci, cname in enumerate(meta.class_names):
        fpr, tpr, _ = roc_curve(y_true_onehot[:, ci], pred_probs[:, ci])
        class_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2, label=f"{cname} (AUC={class_auc:.4f})")
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False positive rate')
    ax.set_ylabel('True positive rate')
    ax.set_title(f'One-vs-rest ROC analysis (run {run_idx + 1}, macro AUC={test_auc_macro:.4f})', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(os.path.join(RESULT_DIR, 'roc_curves_ovr.png'), dpi=300)
    plt.show()
    plt.close(fig)

    # --- Learning curves ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    hist = history.history

    axes[0].plot(hist['loss'], label='Train')
    axes[0].plot(hist['val_loss'], label='Val')
    axes[0].set_title(f'Loss trajectories ({LOSS_NAME})', fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(hist['accuracy'], label='Train')
    axes[1].plot(hist['val_accuracy'], label='Val')
    axes[1].set_title('Accuracy trajectories', fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curves.png'), dpi=300)
    plt.show()
    plt.close(fig)

    # --- Store results for downstream aggregation ---
    all_runs_results.append({
        'run': run_idx + 1,
        'seed': seed,
        'accuracy': test_acc,
        'precision': float(report['weighted avg']['precision']),
        'recall': float(report['weighted avg']['recall']),
        'f1_score': float(report['weighted avg']['f1-score']),
        'auc_ovr_macro': test_auc_macro,
        'auc_ovr_weighted': test_auc_weighted,
        'bal_acc': bal_acc,
        'kappa': kappa,
        'mcc': mcc,
        'per_class_metrics': {
            c: {
                'precision': float(report[c]['precision']),
                'recall': float(report[c]['recall']),
                'f1-score': float(report[c]['f1-score']),
            }
            for c in meta.class_names
        },
        'class_names': list(meta.class_names),
        'result_dir': RESULT_DIR,
        'history': history.history,
        'y_true': y_true_run,
        'y_pred': y_pred_run,
        'pred_probs': pred_probs,
        'test_filenames': meta.test_filenames,
        'n_train': meta.n_train,
        'n_val': meta.n_val,
        'n_test': meta.n_test,
    })

    # Incremental progress table after each run.
    pd.DataFrame([
        {
            'run': r['run'],
            'seed': r['seed'],
            'accuracy': r['accuracy'],
            'precision': r['precision'],
            'recall': r['recall'],
            'f1_score': r['f1_score'],
            'auc_ovr_macro': r['auc_ovr_macro'],
            'auc_ovr_weighted': r['auc_ovr_weighted'],
            'bal_acc': r['bal_acc'],
            'kappa': r['kappa'],
            'mcc': r['mcc'],
        }
        for r in all_runs_results
    ]).to_csv(os.path.join(BASE_RESULT_DIR, 'progress_summary.csv'), index=False)

    print()
    print(f"  Run {run_idx + 1} Results:")
    print(
        f"     Accuracy={test_acc:.4f}  Precision={report['weighted avg']['precision']:.4f}  "
        f"R={report['weighted avg']['recall']:.4f}  F1-score={report['weighted avg']['f1-score']:.4f}"
    )
    print(
        f"     AUC_macro(OvR)={test_auc_macro:.4f}  "
        f"AUC_weighted(OvR)={test_auc_weighted:.4f}"
    )
    print(f"     BalAcc={bal_acc:.4f}  Kappa={kappa:.4f}  MCC={mcc:.4f}")

    # Cleanup loop-local variables.
    try:
        del pred_probs, y_pred_run, y_true_run, y_true_onehot
        del report, cm, hist, history
        del model, train_ds, val_ds, test_ds
    except NameError:
        pass
    tf.keras.backend.clear_session()
    plt.close('all')
    gc.collect()

print()
print("=" * 70)
print(f" ALL {N_RUNS} RUNS COMPLETED")
print("=" * 70)

# Strategy-level summary used by downstream comparison notebooks.
strategy_summary_path = os.path.join(BASE_RESULT_DIR, 'strategy_summary.csv')
pd.DataFrame([
    {
        'strategy_key': STRATEGY_KEY,
        'strategy_label': STRATEGY_LABEL,
        'unfreeze_blocks': str(UNFREEZE_BLOCKS),
        'lr': LR,
        'run': r['run'],
        'seed': r['seed'],
        'accuracy': r['accuracy'],
        'precision': r['precision'],
        'recall': r['recall'],
        'f1_score': r['f1_score'],
        'auc_macro': r['auc_ovr_macro'],
        'auc_weighted': r['auc_ovr_weighted'],
        'bal_acc': r['bal_acc'],
        'kappa': r['kappa'],
        'mcc': r['mcc'],
    }
    for r in all_runs_results
]).to_csv(strategy_summary_path, index=False)
print(f"Saved strategy summary -> {strategy_summary_path}")


In [ ]:
# ========== STANDARDIZED PER-RUN REPORTS AND FIGURES ========== #
# Run this cell after the training loop. It exports the same paper-ready report
# package for every completed seed, not only the manually selected run.
def _metric_rows_for_run(run_data):
    """Build the scalar metric table saved for each run."""
    metric_keys = [
        ("Accuracy", "accuracy"),
        ("Weighted precision", "precision"),
        ("Weighted recall", "recall"),
        ("Weighted F1-score", "f1_score"),
        ("Macro AUC", "auc_macro"),
        ("Weighted AUC", "auc_weighted"),
        ("Macro OvR AUC", "auc_ovr_macro"),
        ("Weighted OvR AUC", "auc_ovr_weighted"),
        ("Balanced accuracy", "bal_acc"),
        ("Cohen's kappa", "kappa"),
        ("Matthews correlation coefficient", "mcc"),
    ]
    rows = []
    for label, key in metric_keys:
        if key in run_data and run_data[key] is not None:
            rows.append({"Metric": label, "Value": float(run_data[key])})
    return rows


def _save_standardized_run_artifacts(run_data):
    """Save standardized test-set reports and diagnostic figures for one run."""
    result_dir = run_data.get("result_dir")
    if not result_dir:
        print("Skipped a run because result_dir is missing.")
        return
    os.makedirs(result_dir, exist_ok=True)

    class_names_local = run_data.get("class_names") or list(run_data.get("per_class_metrics", {}).keys())
    y_true_local = run_data.get("y_true")
    y_pred_local = run_data.get("y_pred")
    pred_probs_local = run_data.get("pred_probs")

    if y_true_local is None or y_pred_local is None or not class_names_local:
        print(f"Skipped standardized artifacts for {result_dir}: missing y_true/y_pred/class_names.")
        return

    y_true_local = np.asarray(y_true_local)
    y_pred_local = np.asarray(y_pred_local)
    run_no = run_data.get("run", "?")
    seed = run_data.get("seed", "?")
    model_label = experiment_display_name()

    # Scalar metric table.
    metric_rows = _metric_rows_for_run(run_data)
    if metric_rows:
        pd.DataFrame(metric_rows).to_csv(
            os.path.join(result_dir, "run_metrics_summary.csv"), index=False)

    # Per-class metric table with readable class labels.
    per_class_metrics = run_data.get("per_class_metrics")
    if not per_class_metrics:
        per_class_metrics = classification_report(
            y_true_local, y_pred_local,
            target_names=class_names_local, output_dict=True, digits=4)
    per_class_rows = []
    for raw_name in class_names_local:
        metrics = per_class_metrics.get(raw_name, {})
        per_class_rows.append({
            "Class": format_class_label(raw_name),
            "Source label": raw_name,
            "Precision": metrics.get("precision", np.nan),
            "Recall": metrics.get("recall", np.nan),
            "F1-score": metrics.get("f1-score", np.nan),
            "Support": metrics.get("support", int(np.sum(y_true_local == class_names_local.index(raw_name))) if raw_name in class_names_local else np.nan),
        })
    pd.DataFrame(per_class_rows).to_csv(
        os.path.join(result_dir, "per_class_metrics_readable.csv"), index=False)

    with open(os.path.join(result_dir, "classification_report_readable.txt"), "w", encoding="utf-8") as f:
        f.write(classification_report(
            y_true_local, y_pred_local,
            target_names=plain_class_names(class_names_local), digits=4))

    # Count and row-normalized confusion matrices.
    cm = confusion_matrix(y_true_local, y_pred_local)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
    labels = display_class_names(class_names_local, width=16)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=axes[0], cbar=True)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=labels, yticklabels=labels, ax=axes[1], cbar=True)
    axes[0].set_title("Confusion matrix counts", fontweight="bold")
    axes[1].set_title("Row-normalized recall matrix", fontweight="bold")
    for axis in axes:
        format_confusion_matrix_axes(axis)
    fig.suptitle(f"{model_label}: test-set confusion matrix analysis for run {run_no} (seed {seed})",
                 fontsize=12, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(os.path.join(result_dir, "confusion_matrix_standardized.png"),
                dpi=300, bbox_inches="tight")
    plt.close(fig)

    # One-vs-rest ROC analysis when probabilities are available.
    if pred_probs_local is not None:
        pred_probs_local = np.asarray(pred_probs_local)
        if pred_probs_local.ndim == 2 and pred_probs_local.shape[1] == len(class_names_local):
            y_bin = label_binarize(y_true_local, classes=range(len(class_names_local)))
            fig, ax = plt.subplots(figsize=(7.5, 6))
            auc_rows = []
            for ci, class_name in enumerate(class_names_local):
                fpr, tpr, _ = roc_curve(y_bin[:, ci], pred_probs_local[:, ci])
                auc_value = auc(fpr, tpr)
                auc_rows.append({"Class": format_class_label(class_name), "AUC": auc_value})
                ax.plot(fpr, tpr, lw=2, label=f"{format_class_label(class_name)} (AUC={auc_value:.3f})")
            ax.plot([0, 1], [0, 1], "--", color="0.55", lw=1)
            ax.set_xlim([0, 1])
            ax.set_ylim([0, 1.01])
            ax.set_xlabel("False positive rate")
            ax.set_ylabel("True positive rate")
            ax.set_title(f"One-vs-rest ROC analysis (run {run_no}, seed {seed})", fontweight="bold")
            ax.legend(loc="lower right", fontsize=8)
            ax.grid(alpha=0.3)
            fig.tight_layout()
            fig.savefig(os.path.join(result_dir, "roc_curves_ovr_standardized.png"),
                        dpi=300, bbox_inches="tight")
            plt.close(fig)
            pd.DataFrame(auc_rows).to_csv(os.path.join(result_dir, "auc_scores_readable.csv"), index=False)

    # Learning curves from the stored Keras history.
    hist = run_data.get("history") or {}
    if "loss" in hist:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
        axes[0].plot(hist.get("loss", []), label="Training loss", lw=2)
        if "val_loss" in hist:
            axes[0].plot(hist.get("val_loss", []), label="Validation loss", lw=2)
        axes[0].set_title("Loss trajectories", fontweight="bold")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        axes[1].plot(hist.get("accuracy", []), label="Training accuracy", lw=2)
        if "val_accuracy" in hist:
            axes[1].plot(hist.get("val_accuracy", []), label="Validation accuracy", lw=2)
        axes[1].set_title("Accuracy trajectories", fontweight="bold")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy")
        axes[1].legend()
        axes[1].grid(alpha=0.3)
        fig.suptitle(f"{model_label}: learning curves for run {run_no} (seed {seed})",
                     fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.90])
        fig.savefig(os.path.join(result_dir, "learning_curves_standardized.png"),
                    dpi=300, bbox_inches="tight")
        plt.close(fig)

    summary_lines = [
        "=" * 80,
        "STANDARDIZED SINGLE-RUN EVALUATION REPORT",
        "=" * 80,
        f"Model/strategy: {model_label}",
        f"Run: {run_no}",
        f"Seed: {seed}",
        f"Result directory: {result_dir}",
        f"Classes: {', '.join(plain_class_names(class_names_local))}",
        "",
        "TEST-SET METRICS",
        "-" * 80,
    ]
    for row in metric_rows:
        summary_lines.append(f"{row['Metric']}: {row['Value']:.4f}")
    summary_lines.extend([
        "",
        "GENERATED ARTIFACTS",
        "-" * 80,
        "run_metrics_summary.csv",
        "per_class_metrics_readable.csv",
        "classification_report_readable.txt",
        "confusion_matrix_standardized.png",
        "roc_curves_ovr_standardized.png (when probability outputs are available)",
        "learning_curves_standardized.png (when history is available)",
    ])
    with open(os.path.join(result_dir, "RUN_EVALUATION_REPORT.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(summary_lines))


for _run_data in all_runs_results:
    _save_standardized_run_artifacts(_run_data)

print(f"Standardized per-run artifacts exported for {len(all_runs_results)} completed run(s).")


In [ ]:
# ========== PAPER-READY ARTIFACT EXPORT SUITE ========== #
# Canonical outputs for thesis/paper writing are saved under:
#   BASE_RESULT_DIR/paper_artifacts/
# Legacy files are left untouched for backward compatibility.

PAPER_ARTIFACT_DIR = os.path.join(BASE_RESULT_DIR, "paper_artifacts")
os.makedirs(PAPER_ARTIFACT_DIR, exist_ok=True)


def _apply_paper_style():
    """Apply a consistent visual style for publication-ready figures."""
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.0)
    plt.rcParams.update({
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8,
        "axes.titleweight": "bold",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.family": "DejaVu Sans",
    })


def _paper_metric_value(run_data, *keys):
    """Return the first available metric value from a run dictionary."""
    for key in keys:
        value = run_data.get(key)
        if value is not None:
            try:
                return float(value)
            except (TypeError, ValueError):
                return value
    return np.nan


def _paper_run_label(run_data):
    """Return a compact run label used in tables and figure legends."""
    return f"Run {run_data.get('run', '?')} (seed {run_data.get('seed', '?')})"


def _paper_run_dir(run_data):
    """Return the canonical artifact directory for one run."""
    run_no = int(run_data.get("run", 0) or 0)
    seed = run_data.get("seed", "unknown")
    out_dir = os.path.join(PAPER_ARTIFACT_DIR, f"run_{run_no:02d}_seed_{seed}")
    os.makedirs(out_dir, exist_ok=True)
    return out_dir


def _paper_safe_array(value):
    """Convert stored numpy/list values to an array when available."""
    if value is None:
        return None
    return np.asarray(value)


def _paper_metrics_dataframe(runs):
    """Build the canonical run-level test metric table."""
    rows = []
    for run_data in runs:
        rows.append({
            "Run": run_data.get("run"),
            "Seed": run_data.get("seed"),
            "Accuracy": _paper_metric_value(run_data, "accuracy"),
            "Weighted precision": _paper_metric_value(run_data, "precision"),
            "Weighted recall": _paper_metric_value(run_data, "recall"),
            "Weighted F1-score": _paper_metric_value(run_data, "f1_score"),
            "Macro AUC": _paper_metric_value(run_data, "auc_macro", "auc_ovr_macro"),
            "Weighted AUC": _paper_metric_value(run_data, "auc_weighted", "auc_ovr_weighted"),
            "Balanced accuracy": _paper_metric_value(run_data, "bal_acc"),
            "Cohen's kappa": _paper_metric_value(run_data, "kappa"),
            "Matthews correlation coefficient": _paper_metric_value(run_data, "mcc"),
        })
    df = pd.DataFrame(rows)
    metric_cols = [c for c in df.columns if c not in ["Run", "Seed"]]
    for col in metric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _paper_metrics_summary(metrics_df):
    """Summarize run-level metrics as mean, SD, and number of valid runs."""
    rows = []
    for col in [c for c in metrics_df.columns if c not in ["Run", "Seed"]]:
        values = pd.to_numeric(metrics_df[col], errors="coerce").dropna()
        if len(values) == 0:
            continue
        rows.append({
            "Metric": col,
            "Mean": float(values.mean()),
            "SD": float(values.std(ddof=0)),
            "N": int(len(values)),
        })
    return pd.DataFrame(rows)


def _paper_per_class_dataframe(runs):
    """Build the canonical per-class metric table across all runs."""
    rows = []
    for run_data in runs:
        class_names_local = run_data.get("class_names") or list(run_data.get("per_class_metrics", {}).keys())
        metrics = run_data.get("per_class_metrics", {})
        for class_name in class_names_local:
            class_metrics = metrics.get(class_name, {})
            rows.append({
                "Run": run_data.get("run"),
                "Seed": run_data.get("seed"),
                "Class": format_class_label(class_name),
                "Source label": class_name,
                "Precision": class_metrics.get("precision", np.nan),
                "Recall": class_metrics.get("recall", np.nan),
                "F1-score": class_metrics.get("f1-score", np.nan),
                "Support": class_metrics.get("support", np.nan),
            })
    df = pd.DataFrame(rows)
    for col in ["Precision", "Recall", "F1-score", "Support"]:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _paper_per_class_summary(per_class_df):
    """Summarize per-class precision, recall, and F1-score across runs."""
    rows = []
    if per_class_df.empty:
        return pd.DataFrame(rows)
    for class_name, group in per_class_df.groupby("Class", sort=False):
        for metric in ["Precision", "Recall", "F1-score"]:
            values = pd.to_numeric(group[metric], errors="coerce").dropna()
            if len(values) == 0:
                continue
            rows.append({
                "Class": class_name,
                "Metric": metric,
                "Mean": float(values.mean()),
                "SD": float(values.std(ddof=0)),
                "N": int(len(values)),
            })
    return pd.DataFrame(rows)


def _paper_save_cross_run_metric_figures(metrics_df, summary_df):
    """Save canonical cross-run metric figures."""
    metric_order = [
        "Accuracy", "Weighted precision", "Weighted recall", "Weighted F1-score",
        "Macro AUC", "Weighted AUC", "Balanced accuracy",
    ]
    plot_df = summary_df[summary_df["Metric"].isin(metric_order)].copy()
    if not plot_df.empty:
        plot_df["Metric"] = pd.Categorical(plot_df["Metric"], categories=metric_order, ordered=True)
        plot_df = plot_df.sort_values("Metric")
        fig, ax = plt.subplots(figsize=(9, 5.2))
        x = np.arange(len(plot_df))
        bars = ax.bar(x, plot_df["Mean"], yerr=plot_df["SD"], capsize=4,
                      color="#4C78A8", edgecolor="black", linewidth=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels(plot_df["Metric"], rotation=25, ha="right")
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Score")
        ax.set_title(f"{experiment_display_name()}: test-set metrics (mean +/- SD across runs)")
        for bar, value in zip(bars, plot_df["Mean"]):
            ax.text(bar.get_x() + bar.get_width() / 2, min(value + 0.025, 1.03),
                    f"{value:.3f}", ha="center", va="bottom", fontsize=8)
        fig.tight_layout()
        fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_01_test_metrics_mean_sd.png"),
                    bbox_inches="tight")
        plt.close(fig)

    metric_cols = [c for c in metric_order if c in metrics_df.columns and not metrics_df[c].isna().all()]
    if metric_cols:
        long_df = metrics_df.melt(id_vars=["Run", "Seed"], value_vars=metric_cols,
                                  var_name="Metric", value_name="Score").dropna()
        if not long_df.empty:
            fig, ax = plt.subplots(figsize=(9, 5.2))
            sns.boxplot(data=long_df, x="Metric", y="Score", ax=ax, color="#72B7B2")
            sns.stripplot(data=long_df, x="Metric", y="Score", ax=ax,
                          color="black", size=4, jitter=0.08)
            ax.set_ylim(0, 1.05)
            ax.set_xlabel("")
            ax.set_ylabel("Score")
            ax.set_title(f"{experiment_display_name()}: test-set metric variability across runs")
            ax.tick_params(axis="x", rotation=25)
            for tick in ax.get_xticklabels():
                tick.set_ha("right")
            fig.tight_layout()
            fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_02_test_metrics_distribution.png"),
                        bbox_inches="tight")
            plt.close(fig)


def _paper_save_per_class_f1(per_class_summary):
    """Save canonical per-class F1-score figure."""
    if per_class_summary.empty:
        return
    f1_df = per_class_summary[per_class_summary["Metric"] == "F1-score"].copy()
    if f1_df.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 5.2))
    x = np.arange(len(f1_df))
    bars = ax.bar(x, f1_df["Mean"], yerr=f1_df["SD"], capsize=4,
                  color="#59A14F", edgecolor="black", linewidth=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(["\\n".join(textwrap.wrap(c, 18)) for c in f1_df["Class"]])
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Class")
    ax.set_ylabel("F1-score")
    ax.set_title(f"{experiment_display_name()}: per-class test-set F1-score (mean +/- SD across runs)")
    for bar, value in zip(bars, f1_df["Mean"]):
        ax.text(bar.get_x() + bar.get_width() / 2, min(value + 0.025, 1.03),
                f"{value:.3f}", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_03_per_class_f1_score.png"),
                bbox_inches="tight")
    plt.close(fig)


def _paper_save_pooled_confusion_and_roc(runs):
    """Save pooled confusion matrix and ROC figures across all runs."""
    class_names_local = runs[0].get("class_names") or list(runs[0].get("per_class_metrics", {}).keys())
    if not class_names_local:
        return
    n_classes = len(class_names_local)
    cm_total = np.zeros((n_classes, n_classes), dtype=float)
    y_true_parts = []
    prob_parts = []
    for run_data in runs:
        y_true_local = _paper_safe_array(run_data.get("y_true"))
        y_pred_local = _paper_safe_array(run_data.get("y_pred"))
        if y_true_local is not None and y_pred_local is not None:
            cm_total += confusion_matrix(y_true_local, y_pred_local, labels=range(n_classes)).astype(float)
            y_true_parts.append(y_true_local)
        probs = _paper_safe_array(run_data.get("pred_probs"))
        if probs is not None and probs.ndim == 2 and probs.shape[1] == n_classes:
            prob_parts.append(probs)
    if cm_total.sum() > 0:
        cm_norm = cm_total / np.maximum(cm_total.sum(axis=1, keepdims=True), 1)
        labels = display_class_names(class_names_local, width=16)
        fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
        sns.heatmap(cm_total.astype(int), annot=True, fmt="d", cmap="Blues",
                    xticklabels=labels, yticklabels=labels, ax=axes[0], cbar=True)
        sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                    xticklabels=labels, yticklabels=labels, ax=axes[1], cbar=True)
        axes[0].set_title("Pooled confusion matrix counts")
        axes[1].set_title("Row-normalized recall matrix")
        for axis in axes:
            format_confusion_matrix_axes(axis)
        fig.suptitle(f"{experiment_display_name()}: pooled test-set confusion matrix analysis",
                     fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.90])
        fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_04_pooled_confusion_matrices.png"),
                    bbox_inches="tight")
        plt.close(fig)
        pd.DataFrame(cm_total.astype(int), index=plain_class_names(class_names_local),
                     columns=plain_class_names(class_names_local)).to_csv(
            os.path.join(PAPER_ARTIFACT_DIR, "table_05_pooled_confusion_matrix_counts.csv"))
        pd.DataFrame(cm_norm, index=plain_class_names(class_names_local),
                     columns=plain_class_names(class_names_local)).to_csv(
            os.path.join(PAPER_ARTIFACT_DIR, "table_06_pooled_confusion_matrix_normalized.csv"))

    if y_true_parts and prob_parts and len(y_true_parts) == len(prob_parts):
        all_y_true = np.concatenate(y_true_parts)
        all_probs = np.concatenate(prob_parts)
        y_bin = label_binarize(all_y_true, classes=range(n_classes))
        auc_rows = []
        fig, ax = plt.subplots(figsize=(7.4, 6.0))
        for ci, class_name in enumerate(class_names_local):
            try:
                fpr, tpr, _ = roc_curve(y_bin[:, ci], all_probs[:, ci])
                auc_value = auc(fpr, tpr)
            except ValueError:
                continue
            auc_rows.append({"Class": format_class_label(class_name), "AUC": float(auc_value)})
            ax.plot(fpr, tpr, lw=2, label=f"{format_class_label(class_name)} (AUC={auc_value:.3f})")
        if auc_rows:
            ax.plot([0, 1], [0, 1], "--", color="0.55", lw=1)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1.01)
            ax.set_xlabel("False positive rate")
            ax.set_ylabel("True positive rate")
            ax.set_title(f"{experiment_display_name()}: pooled one-vs-rest ROC analysis")
            ax.legend(loc="lower right", fontsize=8)
            fig.tight_layout()
            fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_05_pooled_roc_curves_ovr.png"),
                        bbox_inches="tight")
            plt.close(fig)
            pd.DataFrame(auc_rows).to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_07_pooled_auc_scores.csv"),
                                          index=False)


def _paper_save_training_convergence(runs):
    """Save canonical training convergence figure across all runs."""
    runs_with_history = [r for r in runs if isinstance(r.get("history"), dict) and "loss" in r.get("history", {})]
    if not runs_with_history:
        return
    palette = sns.color_palette("deep", n_colors=max(3, len(runs_with_history)))
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5))
    best_epochs = []
    best_val_accs = []
    run_labels = []
    for color, run_data in zip(palette, runs_with_history):
        hist = run_data["history"]
        label = _paper_run_label(run_data)
        run_labels.append(label)
        if "val_accuracy" in hist:
            axes[0, 0].plot(hist["val_accuracy"], color=color, lw=2, label=label)
        if "val_loss" in hist:
            axes[0, 1].plot(hist["val_loss"], color=color, lw=2, label=label)
            best_idx = int(np.argmin(hist["val_loss"]))
        else:
            best_idx = int(np.argmin(hist["loss"]))
        best_epochs.append(best_idx + 1)
        best_val_accs.append(float(hist.get("val_accuracy", hist.get("accuracy", [np.nan]))[best_idx]))
        if "accuracy" in hist and "val_accuracy" in hist:
            gap = np.asarray(hist["accuracy"]) - np.asarray(hist["val_accuracy"])
            axes[1, 1].plot(gap, color=color, lw=2, label=label)
    axes[0, 0].set_title("Validation accuracy trajectory")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Accuracy")
    axes[0, 1].set_title("Validation loss trajectory")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Loss")
    axes[1, 0].bar(run_labels, best_epochs, color=palette[:len(best_epochs)], edgecolor="black")
    axes[1, 0].set_title("Best epoch by validation loss")
    axes[1, 0].set_ylabel("Epoch")
    axes[1, 0].tick_params(axis="x", rotation=20)
    axes[1, 1].set_title("Generalization gap (training - validation accuracy)")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Accuracy gap")
    for axis in [axes[0, 0], axes[0, 1], axes[1, 1]]:
        axis.legend(fontsize=7)
    fig.suptitle(f"{experiment_display_name()}: training convergence analysis across runs",
                 fontsize=12, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_06_training_convergence.png"),
                bbox_inches="tight")
    plt.close(fig)
    pd.DataFrame({
        "Run label": run_labels,
        "Best epoch": best_epochs,
        "Validation accuracy at best epoch": best_val_accs,
    }).to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_08_training_convergence.csv"), index=False)


def _paper_save_single_run_artifacts(runs):
    """Save canonical per-run reports and figures for all completed runs."""
    for run_data in runs:
        out_dir = _paper_run_dir(run_data)
        class_names_local = run_data.get("class_names") or list(run_data.get("per_class_metrics", {}).keys())
        y_true_local = _paper_safe_array(run_data.get("y_true"))
        y_pred_local = _paper_safe_array(run_data.get("y_pred"))
        probs_local = _paper_safe_array(run_data.get("pred_probs"))
        metrics_df = _paper_metrics_dataframe([run_data])
        metrics_df.to_csv(os.path.join(out_dir, "table_01_test_metrics.csv"), index=False)
        if class_names_local and y_true_local is not None and y_pred_local is not None:
            with open(os.path.join(out_dir, "report_classification_metrics.txt"), "w", encoding="utf-8") as f:
                f.write(classification_report(
                    y_true_local, y_pred_local,
                    target_names=plain_class_names(class_names_local), digits=4))
            cm = confusion_matrix(y_true_local, y_pred_local, labels=range(len(class_names_local))).astype(float)
            cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
            labels = display_class_names(class_names_local, width=16)
            fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
            sns.heatmap(cm.astype(int), annot=True, fmt="d", cmap="Blues",
                        xticklabels=labels, yticklabels=labels, ax=axes[0], cbar=True)
            sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                        xticklabels=labels, yticklabels=labels, ax=axes[1], cbar=True)
            axes[0].set_title("Confusion matrix counts")
            axes[1].set_title("Row-normalized recall matrix")
            for axis in axes:
                format_confusion_matrix_axes(axis)
            fig.suptitle(f"{experiment_display_name()}: test-set confusion matrix analysis ({_paper_run_label(run_data)})",
                         fontsize=12, fontweight="bold")
            fig.tight_layout(rect=[0, 0, 1, 0.90])
            fig.savefig(os.path.join(out_dir, "figure_01_confusion_matrices.png"), bbox_inches="tight")
            plt.close(fig)
        if class_names_local and y_true_local is not None and probs_local is not None and probs_local.ndim == 2:
            y_bin = label_binarize(y_true_local, classes=range(len(class_names_local)))
            fig, ax = plt.subplots(figsize=(7.4, 6.0))
            auc_rows = []
            for ci, class_name in enumerate(class_names_local):
                try:
                    fpr, tpr, _ = roc_curve(y_bin[:, ci], probs_local[:, ci])
                    auc_value = auc(fpr, tpr)
                except ValueError:
                    continue
                auc_rows.append({"Class": format_class_label(class_name), "AUC": float(auc_value)})
                ax.plot(fpr, tpr, lw=2, label=f"{format_class_label(class_name)} (AUC={auc_value:.3f})")
            if auc_rows:
                ax.plot([0, 1], [0, 1], "--", color="0.55", lw=1)
                ax.set_xlim(0, 1)
                ax.set_ylim(0, 1.01)
                ax.set_xlabel("False positive rate")
                ax.set_ylabel("True positive rate")
                ax.set_title(f"One-vs-rest ROC analysis ({_paper_run_label(run_data)})")
                ax.legend(loc="lower right", fontsize=8)
                fig.tight_layout()
                fig.savefig(os.path.join(out_dir, "figure_02_roc_curves_ovr.png"), bbox_inches="tight")
                plt.close(fig)
                pd.DataFrame(auc_rows).to_csv(os.path.join(out_dir, "table_02_auc_scores.csv"), index=False)
        hist = run_data.get("history")
        if isinstance(hist, dict) and "loss" in hist:
            fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
            axes[0].plot(hist.get("loss", []), label="Training loss", lw=2)
            if "val_loss" in hist:
                axes[0].plot(hist.get("val_loss", []), label="Validation loss", lw=2)
            axes[0].set_title("Loss trajectories")
            axes[0].set_xlabel("Epoch")
            axes[0].set_ylabel("Loss")
            axes[0].legend()
            axes[1].plot(hist.get("accuracy", []), label="Training accuracy", lw=2)
            if "val_accuracy" in hist:
                axes[1].plot(hist.get("val_accuracy", []), label="Validation accuracy", lw=2)
            axes[1].set_title("Accuracy trajectories")
            axes[1].set_xlabel("Epoch")
            axes[1].set_ylabel("Accuracy")
            axes[1].legend()
            fig.suptitle(f"{experiment_display_name()}: learning curves ({_paper_run_label(run_data)})",
                         fontsize=12, fontweight="bold")
            fig.tight_layout(rect=[0, 0, 1, 0.90])
            fig.savefig(os.path.join(out_dir, "figure_03_learning_curves.png"), bbox_inches="tight")
            plt.close(fig)
        report_lines = [
            "PAPER-READY SINGLE-RUN EVALUATION REPORT",
            "=" * 80,
            f"Model/strategy: {experiment_display_name()}",
            f"Run: {run_data.get('run')}",
            f"Seed: {run_data.get('seed')}",
            f"Source result directory: {run_data.get('result_dir')}",
            f"Canonical artifact directory: {out_dir}",
            "",
            "Generated files:",
            "- table_01_test_metrics.csv",
            "- report_classification_metrics.txt",
            "- figure_01_confusion_matrices.png",
            "- figure_02_roc_curves_ovr.png when probability outputs are available",
            "- figure_03_learning_curves.png when training history is available",
        ]
        with open(os.path.join(out_dir, "report_single_run_evaluation.txt"), "w", encoding="utf-8") as f:
            f.write("\\n".join(report_lines))


def export_paper_ready_artifacts(runs):
    """Export canonical tables, figures, and reports for paper writing."""
    if not runs:
        raise RuntimeError("all_runs_results is empty. Execute the training loop first.")
    _apply_paper_style()
    metrics_df = _paper_metrics_dataframe(runs)
    summary_df = _paper_metrics_summary(metrics_df)
    per_class_df = _paper_per_class_dataframe(runs)
    per_class_summary = _paper_per_class_summary(per_class_df)

    metrics_df.to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_01_test_metrics_by_run.csv"), index=False)
    summary_df.to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_02_test_metrics_summary.csv"), index=False)
    per_class_df.to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_03_per_class_metrics_by_run.csv"), index=False)
    per_class_summary.to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_04_per_class_metrics_summary.csv"), index=False)

    _paper_save_cross_run_metric_figures(metrics_df, summary_df)
    _paper_save_per_class_f1(per_class_summary)
    _paper_save_pooled_confusion_and_roc(runs)
    _paper_save_training_convergence(runs)
    _paper_save_single_run_artifacts(runs)

    report_lines = [
        "PAPER-READY MULTI-RUN EVALUATION REPORT",
        "=" * 80,
        f"Model/strategy: {experiment_display_name()}",
        f"Number of runs: {len(runs)}",
        f"Seeds: {[r.get('seed') for r in runs]}",
        f"Canonical artifact directory: {PAPER_ARTIFACT_DIR}",
        "",
        "Canonical tables:",
        "- table_01_test_metrics_by_run.csv",
        "- table_02_test_metrics_summary.csv",
        "- table_03_per_class_metrics_by_run.csv",
        "- table_04_per_class_metrics_summary.csv",
        "- table_05_pooled_confusion_matrix_counts.csv when predictions are available",
        "- table_06_pooled_confusion_matrix_normalized.csv when predictions are available",
        "- table_07_pooled_auc_scores.csv when probability outputs are available",
        "- table_08_training_convergence.csv when training history is available",
        "",
        "Canonical figures:",
        "- figure_01_test_metrics_mean_sd.png",
        "- figure_02_test_metrics_distribution.png",
        "- figure_03_per_class_f1_score.png",
        "- figure_04_pooled_confusion_matrices.png",
        "- figure_05_pooled_roc_curves_ovr.png when probability outputs are available",
        "- figure_06_training_convergence.png when training history is available",
    ]
    if not summary_df.empty:
        report_lines.extend(["", "Metric summary (mean +/- SD):"])
        for _, row in summary_df.iterrows():
            report_lines.append(f"- {row['Metric']}: {row['Mean']:.4f} +/- {row['SD']:.4f} (n={int(row['N'])})")
    with open(os.path.join(PAPER_ARTIFACT_DIR, "report_multi_run_evaluation.txt"), "w", encoding="utf-8") as f:
        f.write("\\n".join(report_lines))
    print(f"Paper-ready artifacts exported -> {PAPER_ARTIFACT_DIR}")


export_paper_ready_artifacts(all_runs_results)


In [ ]:
# ========== REFERENCE-COMPATIBLE REPORT EXPORTS ========== #
# This cell makes this notebook emit the same report/output contract as
# efficientnetb4-ft-b3to7-cbloss-db1-fn-reviewed.ipynb. It generates real
# artifacts whenever the required data are available and writes an explicit
# status placeholder only for optional interpretability artifacts that require
# a model-specific visualization cell.

import json
import zipfile
import shutil
from pathlib import Path

from sklearn.metrics import (
    balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report, confusion_matrix, roc_curve, auc,
)
from sklearn.preprocessing import label_binarize

REFERENCE_OPTIONAL_FIGURES = [
    "gradcam_visualization.png",
    "gradcam_pp.png",
    "gradcam_pp_misclassified.png",
    "se_attention_maps.png",
    "tsne_features.png",
]


def _ref_display_model_name():
    """Return the model/strategy name used in reference-compatible reports."""
    return experiment_display_name() if "experiment_display_name" in globals() else globals().get("STRATEGY_LABEL", "Garlic classification model")


def _ref_plain_labels(class_names_local):
    """Return publication-facing class labels."""
    return plain_class_names(class_names_local) if "plain_class_names" in globals() else [str(c).replace("_", " ") for c in class_names_local]


def _ref_display_labels(class_names_local, width=16):
    """Return wrapped class labels for compact figures."""
    return display_class_names(class_names_local, width=width) if "display_class_names" in globals() else _ref_plain_labels(class_names_local)


def _ref_metric(run_data, *keys):
    """Return the first available scalar metric from run_data."""
    for key in keys:
        value = run_data.get(key)
        if value is not None:
            try:
                return float(value)
            except (TypeError, ValueError):
                return value
    return np.nan


def _ref_get_class_names(run_data=None):
    """Infer class names from run data or stored per-class metrics."""
    if run_data and run_data.get("class_names"):
        return list(run_data["class_names"])
    if all_runs_results and all_runs_results[0].get("class_names"):
        return list(all_runs_results[0]["class_names"])
    if all_runs_results and all_runs_results[0].get("per_class_metrics"):
        return list(all_runs_results[0]["per_class_metrics"].keys())
    raise RuntimeError("Cannot infer class names from all_runs_results.")


def _ref_array(value):
    """Convert a stored value to a numpy array when it exists."""
    if value is None:
        return None
    return np.asarray(value)


def _ref_run_label(run_data):
    """Return a compact run label."""
    return f"Run {run_data.get('run', '?')} (seed {run_data.get('seed', '?')})"


def _ref_save_placeholder_png(path, title, message):
    """Save an explicit status figure for optional artifacts not generated here."""
    if os.path.exists(path):
        return False
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.axis("off")
    ax.text(0.5, 0.62, title, ha="center", va="center", fontsize=13, fontweight="bold")
    ax.text(0.5, 0.40, message, ha="center", va="center", fontsize=10, wrap=True)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return True


def _ref_copy_if_available(result_dir, candidates, target_name):
    """Create a standard alias from the first existing candidate file."""
    target = os.path.join(result_dir, target_name)
    if os.path.exists(target):
        return "exists"
    for name in candidates:
        src = os.path.join(result_dir, name)
        if os.path.exists(src):
            shutil.copy2(src, target)
            return f"copied from {name}"
    return "missing"


def _ref_run_metrics_rows(run_data):
    """Build run-level metric rows using the reference metric names."""
    return {
        "run": run_data.get("run"),
        "seed": run_data.get("seed"),
        "accuracy": _ref_metric(run_data, "accuracy"),
        "precision": _ref_metric(run_data, "precision"),
        "recall": _ref_metric(run_data, "recall"),
        "f1_score": _ref_metric(run_data, "f1_score"),
        "auc_ovr_macro": _ref_metric(run_data, "auc_ovr_macro", "auc_macro"),
        "auc_ovr_weighted": _ref_metric(run_data, "auc_ovr_weighted", "auc_weighted"),
        "bal_acc": _ref_metric(run_data, "bal_acc"),
        "kappa": _ref_metric(run_data, "kappa"),
        "mcc": _ref_metric(run_data, "mcc"),
    }


def _ref_export_run_level_files(run_data):
    """Export reference-compatible files inside one run directory."""
    result_dir = run_data.get("result_dir")
    if not result_dir:
        return []
    os.makedirs(result_dir, exist_ok=True)
    class_names_local = _ref_get_class_names(run_data)
    plain_labels = _ref_plain_labels(class_names_local)
    y_true_local = _ref_array(run_data.get("y_true"))
    y_pred_local = _ref_array(run_data.get("y_pred"))
    pred_probs_local = _ref_array(run_data.get("pred_probs"))
    history_local = run_data.get("history")
    manifest = []

    # Standard raw arrays used by downstream report notebooks.
    if y_true_local is not None:
        np.save(os.path.join(result_dir, "y_true.npy"), y_true_local)
        manifest.append(("y_true.npy", "generated"))
    if y_pred_local is not None:
        np.save(os.path.join(result_dir, "y_pred.npy"), y_pred_local)
        manifest.append(("y_pred.npy", "generated"))
    if pred_probs_local is not None:
        np.save(os.path.join(result_dir, "pred_probs.npy"), pred_probs_local)
        manifest.append(("pred_probs.npy", "generated"))
    if isinstance(history_local, dict):
        np.save(os.path.join(result_dir, "history.npy"), history_local, allow_pickle=True)
        manifest.append(("history.npy", "generated"))
    if run_data.get("test_filenames") is not None:
        with open(os.path.join(result_dir, "test_filenames.json"), "w", encoding="utf-8") as f:
            json.dump(list(run_data["test_filenames"]), f, ensure_ascii=False, indent=2)
        manifest.append(("test_filenames.json", "generated"))

    # Standard model alias.
    model_status = _ref_copy_if_available(
        result_dir,
        ["best_model.keras", "efficientnetb4_best.keras", "densenet121_best.keras", "inceptionv3_best.keras", "mobilenetv2_best.keras", "nasnetmobile_best.keras"],
        "best_model.keras",
    )
    manifest.append(("best_model.keras", model_status))

    if y_true_local is not None and y_pred_local is not None:
        report_text = classification_report(y_true_local, y_pred_local, target_names=plain_labels, digits=4)
        with open(os.path.join(result_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
            f.write(report_text)
        with open(os.path.join(result_dir, "classification_report_readable.txt"), "w", encoding="utf-8") as f:
            f.write(report_text)
        cm = confusion_matrix(y_true_local, y_pred_local, labels=range(len(class_names_local)))
        fig, ax = plt.subplots(figsize=(7.2, 6.2))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=_ref_display_labels(class_names_local),
                    yticklabels=_ref_display_labels(class_names_local), ax=ax)
        if "format_confusion_matrix_axes" in globals():
            format_confusion_matrix_axes(ax)
        else:
            ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
        ax.set_title(f"{_ref_display_model_name()}: test-set confusion matrix ({_ref_run_label(run_data)})")
        fig.tight_layout()
        fig.savefig(os.path.join(result_dir, "confusion_matrix.png"), dpi=300, bbox_inches="tight")
        fig.savefig(os.path.join(result_dir, "confusion_matrix_standardized.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)
        manifest.extend([
            ("classification_report.txt", "generated"),
            ("classification_report_readable.txt", "generated"),
            ("confusion_matrix.png", "generated"),
            ("confusion_matrix_standardized.png", "generated"),
        ])

        # Per-class metrics and top misclassification CSV.
        report_dict = classification_report(y_true_local, y_pred_local, target_names=class_names_local, output_dict=True, digits=4)
        per_rows = []
        for raw_name, label in zip(class_names_local, plain_labels):
            d = report_dict.get(raw_name, {})
            per_rows.append({
                "Class": label,
                "Source label": raw_name,
                "Precision": d.get("precision", np.nan),
                "Recall": d.get("recall", np.nan),
                "F1-Score": d.get("f1-score", np.nan),
                "Support": d.get("support", np.nan),
            })
        pd.DataFrame(per_rows).to_csv(os.path.join(result_dir, "per_class_metrics.csv"), index=False)
        pd.DataFrame(per_rows).to_csv(os.path.join(result_dir, "per_class_metrics_readable.csv"), index=False)
        wrong_rows = []
        wrong_idx = np.where(y_true_local != y_pred_local)[0]
        if pred_probs_local is not None and pred_probs_local.ndim == 2:
            confidences = pred_probs_local[wrong_idx, y_pred_local[wrong_idx]] if len(wrong_idx) else []
            ordered = wrong_idx[np.argsort(-confidences)] if len(wrong_idx) else []
        else:
            ordered = wrong_idx
        for rank, idx in enumerate(ordered[:50], start=1):
            row = {
                "rank": rank,
                "sample_index": int(idx),
                "true_class": plain_labels[int(y_true_local[idx])],
                "pred_class": plain_labels[int(y_pred_local[idx])],
                "true_source_label": class_names_local[int(y_true_local[idx])],
                "pred_source_label": class_names_local[int(y_pred_local[idx])],
            }
            if pred_probs_local is not None and pred_probs_local.ndim == 2:
                row["pred_confidence"] = float(pred_probs_local[idx, y_pred_local[idx]])
                row["true_class_probability"] = float(pred_probs_local[idx, y_true_local[idx]])
            if run_data.get("test_filenames") is not None:
                row["filename"] = list(run_data["test_filenames"])[idx]
            wrong_rows.append(row)
        pd.DataFrame(wrong_rows).to_csv(os.path.join(result_dir, "top5_misclassified_report.csv"), index=False)
        manifest.extend([
            ("per_class_metrics.csv", "generated"),
            ("per_class_metrics_readable.csv", "generated"),
            ("top5_misclassified_report.csv", "generated"),
        ])

    if y_true_local is not None and pred_probs_local is not None and pred_probs_local.ndim == 2:
        y_bin = label_binarize(y_true_local, classes=range(len(class_names_local)))
        fig, ax = plt.subplots(figsize=(7.5, 6.0))
        auc_rows = []
        for ci, label in enumerate(plain_labels):
            try:
                fpr, tpr, _ = roc_curve(y_bin[:, ci], pred_probs_local[:, ci])
                auc_value = auc(fpr, tpr)
            except ValueError:
                continue
            auc_rows.append({"Class": label, "AUC": float(auc_value)})
            ax.plot(fpr, tpr, lw=2, label=f"{label} (AUC={auc_value:.3f})")
        if auc_rows:
            ax.plot([0, 1], [0, 1], "--", color="0.55", lw=1)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
            ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
            ax.set_title(f"One-vs-rest ROC analysis ({_ref_run_label(run_data)})")
            ax.legend(loc="lower right", fontsize=8)
            fig.tight_layout()
            fig.savefig(os.path.join(result_dir, "roc_curves_ovr.png"), dpi=300, bbox_inches="tight")
            fig.savefig(os.path.join(result_dir, "roc_curves_ovr_standardized.png"), dpi=300, bbox_inches="tight")
            plt.close(fig)
            pd.DataFrame(auc_rows).to_csv(os.path.join(result_dir, "auc_scores_readable.csv"), index=False)
            manifest.extend([
                ("roc_curves_ovr.png", "generated"),
                ("roc_curves_ovr_standardized.png", "generated"),
                ("auc_scores_readable.csv", "generated"),
            ])

    if isinstance(history_local, dict) and "loss" in history_local:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
        axes[0].plot(history_local.get("loss", []), label="Training loss", lw=2)
        if "val_loss" in history_local:
            axes[0].plot(history_local.get("val_loss", []), label="Validation loss", lw=2)
        axes[0].set_title("Loss trajectories")
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
        axes[1].plot(history_local.get("accuracy", []), label="Training accuracy", lw=2)
        if "val_accuracy" in history_local:
            axes[1].plot(history_local.get("val_accuracy", []), label="Validation accuracy", lw=2)
        axes[1].set_title("Accuracy trajectories")
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)
        fig.suptitle(f"{_ref_display_model_name()}: learning curves ({_ref_run_label(run_data)})", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.90])
        fig.savefig(os.path.join(result_dir, "learning_curves.png"), dpi=300, bbox_inches="tight")
        fig.savefig(os.path.join(result_dir, "learning_curves_standardized.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)
        manifest.extend([("learning_curves.png", "generated"), ("learning_curves_standardized.png", "generated")])
    else:
        _ref_copy_if_available(result_dir, ["learning_curve.png"], "learning_curves.png")

    # Optional interpretability artifacts: generate honest status placeholders only if missing.
    for fig_name in REFERENCE_OPTIONAL_FIGURES:
        status = "exists"
        if not os.path.exists(os.path.join(result_dir, fig_name)):
            # t-SNE has a common legacy name in several notebooks.
            if fig_name == "tsne_features.png" and os.path.exists(os.path.join(result_dir, "tsne_embedding.png")):
                shutil.copy2(os.path.join(result_dir, "tsne_embedding.png"), os.path.join(result_dir, fig_name))
                status = "copied from tsne_embedding.png"
            else:
                _ref_save_placeholder_png(
                    os.path.join(result_dir, fig_name),
                    fig_name.replace("_", " ").replace(".png", ""),
                    "This optional interpretability artifact is model-specific. Run the dedicated visualization cell to replace this status figure with the real analysis output.",
                )
                status = "status placeholder"
        manifest.append((fig_name, status))

    # Standard report text files.
    metrics_line = json.dumps(_ref_run_metrics_rows(run_data), ensure_ascii=False, indent=2)
    report_lines = [
        "REFERENCE-COMPATIBLE SINGLE-RUN REPORT",
        "=" * 80,
        f"Model/strategy: {_ref_display_model_name()}",
        f"Run: {run_data.get('run')}",
        f"Seed: {run_data.get('seed')}",
        f"Result directory: {result_dir}",
        "",
        "Metrics:",
        metrics_line,
    ]
    for name in ["RUN_EVALUATION_REPORT.txt", "SUMMARY_REPORT.txt"]:
        with open(os.path.join(result_dir, name), "w", encoding="utf-8") as f:
            f.write("\n".join(report_lines))
        manifest.append((name, "generated"))

    pd.DataFrame(manifest, columns=["artifact", "status"]).to_csv(
        os.path.join(result_dir, "reference_artifact_manifest.csv"), index=False)
    return manifest


def _ref_export_dataset_distribution():
    """Generate class distribution across dataset splits CSV/figure compatible with the reference notebook."""
    rows = []
    data_dir = globals().get("DATA_DIR")
    split_names = ["train", "val", "test"]
    if data_dir and os.path.isdir(data_dir):
        for split in split_names:
            split_dir = os.path.join(data_dir, split)
            if not os.path.isdir(split_dir):
                continue
            for class_name in sorted(os.listdir(split_dir)):
                class_dir = os.path.join(split_dir, class_name)
                if os.path.isdir(class_dir):
                    count = sum(
                        1 for fn in os.listdir(class_dir)
                        if fn.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tiff"))
                    )
                    rows.append({"Split": split, "Class": _ref_plain_labels([class_name])[0], "Source label": class_name, "Count": count})
    if not rows and all_runs_results:
        first = all_runs_results[0]
        rows = [
            {"Split": "train", "Class": "All classes", "Source label": "all", "Count": first.get("n_train", np.nan)},
            {"Split": "val", "Class": "All classes", "Source label": "all", "Count": first.get("n_val", np.nan)},
            {"Split": "test", "Class": "All classes", "Source label": "all", "Count": first.get("n_test", np.nan)},
        ]
    dist_df = pd.DataFrame(rows)
    dist_df.to_csv(os.path.join(BASE_RESULT_DIR, "dataset_distribution.csv"), index=False)
    if not dist_df.empty:
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        sns.barplot(data=dist_df, x="Class", y="Count", hue="Split", ax=ax)
        ax.set_title(f"{_ref_display_model_name()}: class distribution across dataset splits")
        ax.set_xlabel("Class"); ax.set_ylabel("Number of images")
        ax.tick_params(axis="x", rotation=25)
        for tick in ax.get_xticklabels(): tick.set_ha("right")
        fig.tight_layout()
        fig.savefig(os.path.join(BASE_RESULT_DIR, "dataset_distribution.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)


def export_reference_compatible_reports():
    """Export the full reference-compatible report contract."""
    if not all_runs_results:
        raise RuntimeError("all_runs_results is empty. Execute the training loop before this cell.")
    os.makedirs(BASE_RESULT_DIR, exist_ok=True)
    class_names_local = _ref_get_class_names(all_runs_results[0])
    plain_labels = _ref_plain_labels(class_names_local)

    # Per-run exports and base progress summaries.
    manifest_rows = []
    for run_data in all_runs_results:
        for artifact, status in _ref_export_run_level_files(run_data):
            manifest_rows.append({
                "run": run_data.get("run"),
                "seed": run_data.get("seed"),
                "artifact": artifact,
                "status": status,
                "result_dir": run_data.get("result_dir"),
            })
    pd.DataFrame(manifest_rows).to_csv(os.path.join(BASE_RESULT_DIR, "reference_artifact_manifest.csv"), index=False)

    progress_df = pd.DataFrame([_ref_run_metrics_rows(r) for r in all_runs_results])
    progress_df.to_csv(os.path.join(BASE_RESULT_DIR, "progress_summary.csv"), index=False)
    progress_df.to_csv(os.path.join(BASE_RESULT_DIR, "strategy_summary.csv"), index=False)
    progress_df.assign(strategy=globals().get("STRATEGY_KEY", _ref_display_model_name())).to_csv(
        os.path.join(BASE_RESULT_DIR, "summary.csv"), index=False)

    # Additional metrics summary.
    additional_rows = []
    for run_data in all_runs_results:
        y_true_local = _ref_array(run_data.get("y_true"))
        y_pred_local = _ref_array(run_data.get("y_pred"))
        row = _ref_run_metrics_rows(run_data)
        if y_true_local is not None and y_pred_local is not None:
            row["bal_acc"] = float(balanced_accuracy_score(y_true_local, y_pred_local))
            row["kappa"] = float(cohen_kappa_score(y_true_local, y_pred_local))
            row["mcc"] = float(matthews_corrcoef(y_true_local, y_pred_local))
        additional_rows.append(row)
    additional_df = pd.DataFrame(additional_rows)
    additional_df.to_csv(os.path.join(BASE_RESULT_DIR, "additional_metrics.csv"), index=False)
    additional_df.describe(include="all").to_csv(os.path.join(BASE_RESULT_DIR, "additional_metrics_summary.csv"))

    _ref_export_dataset_distribution()

    # Pooled confusion matrix and ROC curves across all runs.
    y_true_parts, y_pred_parts, prob_parts = [], [], []
    for run_data in all_runs_results:
        yt = _ref_array(run_data.get("y_true")); yp = _ref_array(run_data.get("y_pred")); pp = _ref_array(run_data.get("pred_probs"))
        if yt is not None and yp is not None:
            y_true_parts.append(yt); y_pred_parts.append(yp)
        if yt is not None and pp is not None and pp.ndim == 2:
            prob_parts.append(pp)
    if y_true_parts and y_pred_parts:
        all_y_true = np.concatenate(y_true_parts)
        all_y_pred = np.concatenate(y_pred_parts)
        cm = confusion_matrix(all_y_true, all_y_pred, labels=range(len(class_names_local)))
        cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=_ref_display_labels(class_names_local), yticklabels=_ref_display_labels(class_names_local), ax=axes[0])
        sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, xticklabels=_ref_display_labels(class_names_local), yticklabels=_ref_display_labels(class_names_local), ax=axes[1])
        axes[0].set_title("Pooled confusion matrix counts"); axes[1].set_title("Row-normalized recall matrix")
        for ax in axes:
            if "format_confusion_matrix_axes" in globals(): format_confusion_matrix_axes(ax)
            else: ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
        fig.suptitle(f"{_ref_display_model_name()}: pooled test-set confusion matrix analysis", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.90])
        fig.savefig(os.path.join(BASE_RESULT_DIR, "aggregate_confusion_matrix.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)

        correct_mask = all_y_true == all_y_pred
        fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.0))
        if prob_parts and len(prob_parts) == len(y_true_parts):
            all_probs = np.concatenate(prob_parts)
            conf = np.max(all_probs, axis=1)
            axes[0].hist(conf[correct_mask], bins=20, alpha=0.7, label="Correct", edgecolor="black")
            axes[0].hist(conf[~correct_mask], bins=20, alpha=0.7, label="Misclassified", edgecolor="black")
        else:
            axes[0].bar(["Correct", "Misclassified"], [int(correct_mask.sum()), int((~correct_mask).sum())])
        axes[0].set_title("Prediction outcome counts"); axes[0].set_ylabel("Count"); axes[0].legend()
        class_acc = []
        for ci, label in enumerate(plain_labels):
            mask = all_y_true == ci
            class_acc.append((label, float(np.mean(all_y_pred[mask] == ci)) if mask.any() else np.nan))
        acc_df = pd.DataFrame(class_acc, columns=["Class", "Accuracy"]).sort_values("Accuracy")
        sns.barplot(data=acc_df, x="Accuracy", y="Class", ax=axes[1], color="#4C78A8")
        axes[1].set_xlim(0, 1.05); axes[1].set_title("Class-wise test accuracy")
        fig.tight_layout()
        fig.savefig(os.path.join(BASE_RESULT_DIR, "error_analysis.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)

    if y_true_parts and prob_parts and len(y_true_parts) == len(prob_parts):
        all_y_true = np.concatenate(y_true_parts)
        all_probs = np.concatenate(prob_parts)
        y_bin = label_binarize(all_y_true, classes=range(len(class_names_local)))
        fig, ax = plt.subplots(figsize=(7.5, 6.0))
        auc_rows = []
        for ci, label in enumerate(plain_labels):
            try:
                fpr, tpr, _ = roc_curve(y_bin[:, ci], all_probs[:, ci])
                auc_value = auc(fpr, tpr)
            except ValueError:
                continue
            auc_rows.append({"Class": label, "AUC": float(auc_value)})
            ax.plot(fpr, tpr, lw=2, label=f"{label} (AUC={auc_value:.3f})")
        if auc_rows:
            ax.plot([0, 1], [0, 1], "--", color="0.55", lw=1)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
            ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
            ax.set_title(f"{_ref_display_model_name()}: pooled one-vs-rest ROC analysis")
            ax.legend(loc="lower right", fontsize=8)
            fig.tight_layout()
            fig.savefig(os.path.join(BASE_RESULT_DIR, "roc_curves.png"), dpi=300, bbox_inches="tight")
            plt.close(fig)
            pd.DataFrame(auc_rows).to_csv(os.path.join(BASE_RESULT_DIR, "auc_scores.csv"), index=False)

    # Training convergence across runs.
    hist_runs = [r for r in all_runs_results if isinstance(r.get("history"), dict) and "loss" in r.get("history", {})]
    if hist_runs:
        fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5))
        palette = sns.color_palette("deep", n_colors=max(3, len(hist_runs)))
        best_epochs = []
        labels = []
        for color, run_data in zip(palette, hist_runs):
            hist = run_data["history"]; label = _ref_run_label(run_data); labels.append(label)
            if "val_accuracy" in hist: axes[0, 0].plot(hist["val_accuracy"], color=color, label=label, lw=2)
            if "val_loss" in hist: axes[0, 1].plot(hist["val_loss"], color=color, label=label, lw=2)
            best_idx = int(np.argmin(hist.get("val_loss", hist.get("loss"))))
            best_epochs.append(best_idx + 1)
            if "accuracy" in hist and "val_accuracy" in hist:
                axes[1, 1].plot(np.asarray(hist["accuracy"]) - np.asarray(hist["val_accuracy"]), color=color, label=label, lw=2)
        axes[0, 0].set_title("Validation accuracy trajectory"); axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Accuracy")
        axes[0, 1].set_title("Validation loss trajectory"); axes[0, 1].set_xlabel("Epoch"); axes[0, 1].set_ylabel("Loss")
        axes[1, 0].bar(labels, best_epochs, color=palette[:len(best_epochs)], edgecolor="black")
        axes[1, 0].set_title("Best epoch by validation loss"); axes[1, 0].tick_params(axis="x", rotation=20)
        axes[1, 1].set_title("Generalization gap (training - validation accuracy)"); axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_ylabel("Accuracy gap")
        for ax in [axes[0,0], axes[0,1], axes[1,1]]: ax.legend(fontsize=7); ax.grid(alpha=0.3)
        fig.suptitle(f"{_ref_display_model_name()}: training convergence analysis across runs", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0,0,1,0.94])
        fig.savefig(os.path.join(BASE_RESULT_DIR, "convergence_analysis.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)

    # Multi-run report files using reference names.
    report_lines = [
        "REFERENCE-COMPATIBLE MULTI-RUN REPORT",
        "=" * 80,
        f"Model/strategy: {_ref_display_model_name()}",
        f"Number of runs: {len(all_runs_results)}",
        f"Seeds: {[r.get('seed') for r in all_runs_results]}",
        f"Base result directory: {BASE_RESULT_DIR}",
        "",
        "Generated reference-compatible artifacts are listed in reference_artifact_manifest.csv.",
    ]
    with open(os.path.join(BASE_RESULT_DIR, "MULTI_RUN_SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    # Base qualitative archive when a qualitative_analysis directory exists.
    selected_result_dir = all_runs_results[-1].get("result_dir")
    if selected_result_dir:
        qa_dir = os.path.join(selected_result_dir, "qualitative_analysis")
        os.makedirs(qa_dir, exist_ok=True)
        with open(os.path.join(qa_dir, "analysis_notes.txt"), "w", encoding="utf-8") as f:
            f.write("Reference-compatible qualitative analysis directory. Replace placeholder figures by running dedicated visualization cells when available.\n")
        zip_path = os.path.join(BASE_RESULT_DIR, "qualitative_analysis.zip")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for root_dir, _, files in os.walk(qa_dir):
                for filename in files:
                    full = os.path.join(root_dir, filename)
                    zf.write(full, arcname=os.path.relpath(full, qa_dir))

    print(f"Reference-compatible report contract exported -> {BASE_RESULT_DIR}")


export_reference_compatible_reports()


---
## Section 2 ? Results Aggregation & Scientific Reports

Aggregate metrics across all runs, generate LaTeX tables, CSV summaries, and visualizations for publication.


In [ ]:
# ============================================================================
# CELL 8: SCIENTIFIC REPORTING SUITE (THESIS + PAPER)
# ============================================================================
# Produces publication-ready tables, figures, and text reports from multi-run
# results. Outputs are saved in BASE_RESULT_DIR.
# ============================================================================
if len(all_runs_results) == 0:
    raise RuntimeError('all_runs_results is empty. Execute the training loop first.')

print("\n" + "=" * 80)
print("SCIENTIFIC REPORTING SUITE")
print("=" * 80 + "\n")

# ---------------------------------------------------------------------------
# 1) Aggregate metrics across runs
# ---------------------------------------------------------------------------
accuracies = [r['accuracy'] for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls = [r['recall'] for r in all_runs_results]
f1_scores = [r['f1_score'] for r in all_runs_results]
auc_macro_scores = [r['auc_ovr_macro'] for r in all_runs_results]
auc_weighted_scores = [r['auc_ovr_weighted'] for r in all_runs_results]
bal_accs = [r['bal_acc'] for r in all_runs_results]
kappas = [r['kappa'] for r in all_runs_results]
mccs = [r['mcc'] for r in all_runs_results]

overall_stats = {
    'Accuracy': {'mean': np.mean(accuracies), 'std': np.std(accuracies), 'values': accuracies},
    'Precision': {'mean': np.mean(precisions), 'std': np.std(precisions), 'values': precisions},
    'Recall': {'mean': np.mean(recalls), 'std': np.std(recalls), 'values': recalls},
    'F1-Score': {'mean': np.mean(f1_scores), 'std': np.std(f1_scores), 'values': f1_scores},
    'AUC Macro': {'mean': np.mean(auc_macro_scores), 'std': np.std(auc_macro_scores), 'values': auc_macro_scores},
    'AUC-Weighted': {'mean': np.mean(auc_weighted_scores), 'std': np.std(auc_weighted_scores), 'values': auc_weighted_scores},
    'Balanced Accuracy': {'mean': np.mean(bal_accs), 'std': np.std(bal_accs), 'values': bal_accs},
    'Cohen Kappa': {'mean': np.mean(kappas), 'std': np.std(kappas), 'values': kappas},
    'MCC': {'mean': np.mean(mccs), 'std': np.std(mccs), 'values': mccs},
}

print("OVERALL METRICS ACROSS ALL RUNS")
print("-" * 80)
for metric_name in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC Macro', 'AUC-Weighted']:
    stats = overall_stats[metric_name]
    print(f"{metric_name:16s}: {stats['mean']:.4f} +/- {stats['std']:.4f}")
    print(f"                  Per-run: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 80)

class_names = list(all_runs_results[0]['per_class_metrics'].keys())
per_class_stats = {}
for class_name in class_names:
    per_class_stats[class_name] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][class_name][metric] for r in all_runs_results]
        per_class_stats[class_name][metric] = {
            'mean': np.mean(values),
            'std': np.std(values),
            'values': values,
        }

print("\nStatistics aggregated successfully.")

# ---------------------------------------------------------------------------
# 2) Scientific summary tables (CSV + console)
# ---------------------------------------------------------------------------
run_cols = [f"Run {i + 1}" for i in range(len(all_runs_results))]
metric_order = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC Macro', 'AUC-Weighted']

overall_df = pd.DataFrame({
    'Metric': metric_order,
    'Mean': [overall_stats[m]['mean'] for m in metric_order],
    'Std': [overall_stats[m]['std'] for m in metric_order],
})
for i, col in enumerate(run_cols):
    overall_df[col] = [overall_stats[m]['values'][i] for m in metric_order]

overall_df['Mean +/- SD'] = overall_df.apply(
    lambda row: f"{row['Mean']:.4f} +/- {row['Std']:.4f}", axis=1
)

overall_df.to_csv(os.path.join(BASE_RESULT_DIR, 'overall_metrics_summary.csv'), index=False)

variant_results_path = os.path.join(BASE_RESULT_DIR, RESULTS_CSV_NAME)
overall_df[['Metric', 'Mean +/- SD'] + run_cols].to_csv(variant_results_path, index=False)
print(f"Saved variant results -> {variant_results_path}")

per_class_rows = []
for class_name in class_names:
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        row = {
            'Class': class_name,
            'Metric': metric.capitalize(),
            'Mean': stats['mean'],
            'Std': stats['std'],
            'Mean +/- SD': f"{stats['mean']:.4f} +/- {stats['std']:.4f}",
        }
        for i, col in enumerate(run_cols):
            row[col] = stats['values'][i]
        per_class_rows.append(row)

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(os.path.join(BASE_RESULT_DIR, 'per_class_metrics_summary.csv'), index=False)

print("\nTEST-SET CLASSIFICATION PERFORMANCE")
print("=" * 80)
print(overall_df[['Metric', 'Mean +/- SD'] + run_cols].to_string(index=False))
print("=" * 80)

print("\nPER-CLASS CLASSIFICATION PERFORMANCE")
print("=" * 80)
for class_name in class_names:
    class_data = per_class_df[per_class_df['Class'] == class_name]
    print(f"\n{class_name}:")
    print(class_data[['Metric', 'Mean +/- SD']].to_string(index=False))
print("=" * 80)

# ---------------------------------------------------------------------------
# 3) LaTeX tables for manuscript
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("GENERATING LATEX TABLES")
print("=" * 80 + "\n")

latex_overall_df = overall_df[['Metric', 'Mean +/- SD'] + run_cols].copy()
latex_overall = latex_overall_df.to_latex(
    index=False,
    escape=False,
    caption=f"Test-set classification performance for V1 EfficientNetB4 GAP+GMP + CBLoss (mean +/- SD across {len(all_runs_results)} independent runs)",
    label='tab:gdenet_ablation_overall',
)

per_class_compact_rows = []
for class_name in class_names:
    prec = per_class_stats[class_name]['precision']
    rec = per_class_stats[class_name]['recall']
    f1 = per_class_stats[class_name]['f1-score']
    per_class_compact_rows.append({
        'Class': class_name,
        'Precision': f"{prec['mean']:.4f} +/- {prec['std']:.4f}",
        'Recall': f"{rec['mean']:.4f} +/- {rec['std']:.4f}",
        'F1-Score': f"{f1['mean']:.4f} +/- {f1['std']:.4f}",
    })
per_class_compact_df = pd.DataFrame(per_class_compact_rows)

latex_per_class = per_class_compact_df.to_latex(
    index=False,
    escape=False,
    caption='Per-class classification performance for V1 EfficientNetB4 GAP+GMP + CBLoss (mean +/- SD across all independent runs)',
    label='tab:gdenet_ablation_per_class',
)

with open(os.path.join(BASE_RESULT_DIR, 'latex_tables.tex'), 'w', encoding='utf-8') as f:
    f.write('% Overall Metrics Table\n')
    f.write(latex_overall)
    f.write('\n\n% Per-Class Metrics Table\n')
    f.write(latex_per_class)

print("Latex tables saved to 'latex_tables.tex'.")

# ---------------------------------------------------------------------------
# 4) Publication figures
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("CREATING FIGURES")
print("=" * 80 + "\n")

fig, ax = plt.subplots(figsize=(11, 6))
plot_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC Macro', 'AUC-Weighted']
means = [overall_stats[m]['mean'] for m in plot_metrics]
stds = [overall_stats[m]['std'] for m in plot_metrics]

x_pos = np.arange(len(plot_metrics))
ax.bar(x_pos, means, yerr=stds, capsize=8, alpha=0.85, color='steelblue', edgecolor='black')
ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title(f'{experiment_display_name()}: test-set metrics (mean +/- SD across runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(plot_metrics, rotation=20, ha='right')
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, min(1.03, m + s + 0.01), f"{m:.4f}\n+/-{s:.4f}", ha='center', va='bottom', fontsize=8)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'overall_metrics_barplot.png'), dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(11, 6))
data_for_box = [accuracies, precisions, recalls, f1_scores, auc_macro_scores, auc_weighted_scores]
bp = ax.boxplot(data_for_box, labels=plot_metrics, patch_artist=True, showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='red', markersize=7))
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
    patch.set_alpha(0.7)
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Run-to-run variability of test-set metrics', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'metrics_boxplot.png'), dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
class_f1_means = [per_class_stats[c]['f1-score']['mean'] for c in class_names]
class_f1_stds = [per_class_stats[c]['f1-score']['std'] for c in class_names]
x_pos = np.arange(len(class_names))
ax.bar(x_pos, class_f1_means, yerr=class_f1_stds, capsize=5, alpha=0.85, color='coral', edgecolor='black')
ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('Per-class test-set F1-score (mean +/- SD across runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(display_class_names(class_names, width=16), rotation=35, ha='right')
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'per_class_f1score.png'), dpi=300, bbox_inches='tight')
plt.show()

# ---------------------------------------------------------------------------
# 5) Aggregate confusion matrix across runs
# ---------------------------------------------------------------------------
cls = all_runs_results[0]['class_names']
n_cls = len(cls)
agg_cm = np.zeros((n_cls, n_cls), dtype=float)
for r in all_runs_results:
    agg_cm += confusion_matrix(r['y_true'], r['y_pred']).astype(float)

row_sum = agg_cm.sum(axis=1, keepdims=True)
agg_cm_norm = np.divide(agg_cm, row_sum, out=np.zeros_like(agg_cm), where=row_sum != 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(agg_cm.astype(int), annot=True, fmt='d',
            xticklabels=display_class_names(cls), yticklabels=display_class_names(cls), cmap='Blues', ax=axes[0])
axes[0].set_title('Pooled confusion matrix counts', fontweight='bold')
format_confusion_matrix_axes(axes[0])

sns.heatmap(agg_cm_norm, annot=True, fmt='.2%',
            xticklabels=display_class_names(cls), yticklabels=display_class_names(cls), cmap='Blues', ax=axes[1])
axes[1].set_title('Row-normalized recall matrix', fontweight='bold')
format_confusion_matrix_axes(axes[1])

plt.suptitle(f'{experiment_display_name()}: pooled test-set confusion matrix analysis across runs', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'aggregate_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

# ---------------------------------------------------------------------------
# 6) Aggregate ROC curves and AUC summary
# ---------------------------------------------------------------------------
all_y_true = np.concatenate([r['y_true'] for r in all_runs_results])
all_probs = np.concatenate([r['pred_probs'] for r in all_runs_results])
y_bin = label_binarize(all_y_true, classes=range(n_cls))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
tab_colors = plt.cm.tab10.colors
auc_scores = {}
fpr_d, tpr_d = {}, {}

for i, cname in enumerate(cls):
    fpr_d[i], tpr_d[i], _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr_d[i], tpr_d[i])
    auc_scores[cname] = roc_auc
    axes[0].plot(fpr_d[i], tpr_d[i], color=tab_colors[i % len(tab_colors)], lw=2,
                 label=f'{cname} (AUC={roc_auc:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set(xlim=[0, 1], ylim=[0, 1.01], xlabel='False positive rate', ylabel='True positive rate',
            title='Per-class one-vs-rest ROC analysis aggregated across runs')
axes[0].legend(loc='lower right', fontsize=8)
axes[0].grid(alpha=0.3)

all_fpr = np.unique(np.concatenate([fpr_d[i] for i in range(n_cls)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_cls):
    mean_tpr += np.interp(all_fpr, fpr_d[i], tpr_d[i])
mean_tpr /= n_cls
macro_auc = auc(all_fpr, mean_tpr)

axes[1].plot(all_fpr, mean_tpr, 'b-', lw=2, label=f'Macro-avg (AUC={macro_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set(xlim=[0, 1], ylim=[0, 1.01], xlabel='False positive rate', ylabel='True positive rate',
            title='Macro-average ROC curve')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle(f'{experiment_display_name()}: pooled one-vs-rest ROC analysis across runs', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

weights = [np.sum(all_y_true == i) for i in range(n_cls)]
weighted_auc = np.average(list(auc_scores.values()), weights=weights)

auc_df = pd.DataFrame({'Class': list(auc_scores.keys()), 'AUC': list(auc_scores.values())})
auc_df = pd.concat([
    auc_df,
    pd.DataFrame([
        {'Class': 'Macro-Average', 'AUC': macro_auc},
        {'Class': 'Weighted-Average', 'AUC': weighted_auc},
    ]),
], ignore_index=True)
auc_df.to_csv(os.path.join(BASE_RESULT_DIR, 'auc_scores.csv'), index=False)

# ---------------------------------------------------------------------------
# 7) Class distribution across dataset splits and additional metrics tables
# ---------------------------------------------------------------------------
splits = ['train', 'val', 'test']
split_counts = {}
for split in splits:
    split_dir = os.path.join(DATA_DIR, split)
    counts = {
        cls_name: len(glob.glob(os.path.join(split_dir, cls_name, '*')))
        for cls_name in sorted(os.listdir(split_dir))
        if os.path.isdir(os.path.join(split_dir, cls_name))
    }
    split_counts[split] = counts

dist_df = pd.DataFrame(split_counts).fillna(0).astype(int)
dist_df['Total'] = dist_df.sum(axis=1)
dist_df.to_csv(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.csv'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dist_df[splits].plot(kind='bar', ax=axes[0], color=['steelblue', 'coral', 'seagreen'], edgecolor='black', width=0.7)
axes[0].set_title('Class distribution across dataset splits', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Images')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(title='Split')
axes[0].grid(axis='y', alpha=0.3)

axes[1].pie(dist_df['Total'], labels=dist_df.index, autopct='%1.1f%%', startangle=140,
            colors=plt.cm.Set3.colors[:len(dist_df)])
axes[1].set_title('Overall class frequency distribution', fontsize=13, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()

extra_df = pd.DataFrame([
    {'Run': r['run'], 'Seed': r['seed'], 'Balanced Accuracy': r['bal_acc'], 'Kappa': r['kappa'], 'MCC': r['mcc']}
    for r in all_runs_results
])
mean_row = {'Run': 'Mean', 'Seed': '-',
            'Balanced Accuracy': extra_df['Balanced Accuracy'].mean(),
            'Kappa': extra_df['Kappa'].mean(),
            'MCC': extra_df['MCC'].mean()}
std_row = {'Run': 'Std', 'Seed': '-',
           'Balanced Accuracy': extra_df['Balanced Accuracy'].std(ddof=0),
           'Kappa': extra_df['Kappa'].std(ddof=0),
           'MCC': extra_df['MCC'].std(ddof=0)}
summary_extra = pd.concat([extra_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

extra_df.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics.csv'), index=False)
summary_extra.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics_summary.csv'), index=False)

# ---------------------------------------------------------------------------
# 8) Convergence analysis
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['steelblue', 'coral', 'seagreen', 'olive', 'purple']

for i, r in enumerate(all_runs_results):
    c = colors[i % len(colors)]
    axes[0, 0].plot(r['history']['val_accuracy'], color=c, label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 0].set_title('Validation accuracy trajectories across runs', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

for i, r in enumerate(all_runs_results):
    c = colors[i % len(colors)]
    axes[0, 1].plot(r['history']['val_loss'], color=c, label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 1].set_title('Validation loss trajectories across runs', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

best_epochs = [int(np.argmin(r['history']['val_loss']) + 1) for r in all_runs_results]
run_labels = [f"Run {r['run']}\n(seed={r['seed']})" for r in all_runs_results]
best_val_accs = [float(r['history']['val_accuracy'][np.argmin(r['history']['val_loss'])]) for r in all_runs_results]

bars = axes[1, 0].bar(run_labels, best_epochs, color=colors[:len(all_runs_results)], edgecolor='black', alpha=0.85)
axes[1, 0].set_title('Best epoch by validation loss per run', fontweight='bold')
axes[1, 0].set_ylabel('Epoch')
axes[1, 0].grid(axis='y', alpha=0.3)
for bar, ep in zip(bars, best_epochs):
    axes[1, 0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    str(ep), ha='center', fontweight='bold')

for i, r in enumerate(all_runs_results):
    c = colors[i % len(colors)]
    gap = np.array(r['history']['accuracy']) - np.array(r['history']['val_accuracy'])
    axes[1, 1].plot(gap, color=c, label=f"Run {r['run']}", lw=2)
axes[1, 1].axhline(0, color='black', linestyle='--', lw=1)
axes[1, 1].set_title('Generalization gap (training - validation accuracy)', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Train Acc - Val Acc')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.suptitle(f'{experiment_display_name()}: training convergence analysis across runs', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'convergence_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

# ---------------------------------------------------------------------------
# 9) Comprehensive text report + baseline comparison + archive
# ---------------------------------------------------------------------------
report_lines = []
report_lines.append('=' * 100)
report_lines.append('V1 EfficientNetB4 GAP+GMP + CBLoss MULTI-RUN EXPERIMENT REPORT')
report_lines.append('=' * 100)
report_lines.append(f"\nGenerated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append(f"Strategy: {STRATEGY_LABEL}")
report_lines.append(f"Strategy key: {STRATEGY_KEY}")
report_lines.append(f"Random Seeds: {RANDOM_SEEDS[:N_RUNS]}")
report_lines.append(f"Number of Runs: {len(all_runs_results)}")
report_lines.append('')

report_lines.append('=' * 100)
report_lines.append('TEST-SET CLASSIFICATION PERFORMANCE (Mean +/- SD)')
report_lines.append('=' * 100)
report_lines.append(overall_df[['Metric', 'Mean +/- SD'] + run_cols].to_string(index=False))

report_lines.append('\n' + '=' * 100)
report_lines.append('PER-CLASS CLASSIFICATION PERFORMANCE (Mean +/- SD)')
report_lines.append('=' * 100)
for class_name in class_names:
    report_lines.append(f"\nClass: {class_name}")
    class_data = per_class_df[per_class_df['Class'] == class_name]
    report_lines.append(class_data[['Metric', 'Mean +/- SD']].to_string(index=False))

report_lines.append('\n' + '=' * 100)
report_lines.append('INDIVIDUAL RUN DETAILS')
report_lines.append('=' * 100)
for r in all_runs_results:
    report_lines.append(f"\nRun {r['run']} (Seed: {r['seed']})")
    report_lines.append(f"  Accuracy      : {r['accuracy']:.4f}")
    report_lines.append(f"  Precision     : {r['precision']:.4f}")
    report_lines.append(f"  Recall        : {r['recall']:.4f}")
    report_lines.append(f"  F1-Score      : {r['f1_score']:.4f}")
    report_lines.append(f"  AUC Macro     : {r['auc_ovr_macro']:.4f}")
    report_lines.append(f"  AUC-Weighted  : {r['auc_ovr_weighted']:.4f}")
    report_lines.append(f"  Balanced Acc  : {r['bal_acc']:.4f}")
    report_lines.append(f"  Kappa         : {r['kappa']:.4f}")
    report_lines.append(f"  MCC           : {r['mcc']:.4f}")
    report_lines.append(f"  Result Dir    : {r['result_dir']}")

baseline_acc = 0.92
proposed_acc = np.mean(accuracies)
diff = proposed_acc - baseline_acc
report_lines.append('\n' + '=' * 100)
report_lines.append('BASELINE COMPARISON')
report_lines.append('=' * 100)
report_lines.append(f"Baseline (FSDA): ~{baseline_acc:.4f}")
report_lines.append(f"Current (V1 EfficientNetB4 GAP+GMP + CBLoss): {proposed_acc:.4f} +/- {np.std(accuracies):.4f}")
report_lines.append(f"Difference: {diff:+.4f} ({'IMPROVED' if diff > 0 else 'NEEDS TUNING'})")

report_text = "\n".join(report_lines)
print("\n" + report_text)

with open(os.path.join(BASE_RESULT_DIR, 'MULTI_RUN_SUMMARY_REPORT.txt'), 'w', encoding='utf-8') as f:
    f.write(report_text)

# Legacy summary file kept for compatibility with previous workflows.
summary_df = pd.DataFrame([
    {
        'strategy': STRATEGY_KEY,
        'run': r['run'],
        'seed': r['seed'],
        'accuracy': r['accuracy'],
        'precision': r['precision'],
        'recall': r['recall'],
        'f1_score': r['f1_score'],
        'auc_ovr_macro': r['auc_ovr_macro'],
        'auc_ovr_weighted': r['auc_ovr_weighted'],
        'bal_acc': r['bal_acc'],
        'kappa': r['kappa'],
        'mcc': r['mcc'],
    }
    for r in all_runs_results
])
summary_df.to_csv(os.path.join(BASE_RESULT_DIR, 'summary.csv'), index=False)

zip_path = f"/kaggle/working/{STRATEGY_KEY}_complete"
shutil.make_archive(zip_path, 'zip', BASE_RESULT_DIR)
print("\nArchived -> " + zip_path + '.zip')
print('All thesis/paper reporting artifacts have been generated.')


---
## Section 3 — Advanced Analysis for Thesis / Paper

Reports below aggregate data from **all runs** (dataset, confusion matrix, ROC curves, Kappa/MCC) and then provide **single-run deep-dive** analysis (Grad-CAM, t-SNE, error analysis).

> **To analyse a specific run:** change `SELECTED_RUN` in the *Select Run* cell below (1 / 2 / 3).


In [ ]:
# ========== DATASET DISTRIBUTION ANALYSIS ========== #
splits = ['train', 'val', 'test']
split_counts = {}
for split in splits:
    split_dir = os.path.join(DATA_DIR, split)
    counts = {cls: len(os.listdir(os.path.join(split_dir, cls)))
              for cls in sorted(os.listdir(split_dir))
              if os.path.isdir(os.path.join(split_dir, cls))}
    split_counts[split] = counts

dist_df = pd.DataFrame(split_counts).fillna(0).astype(int)
dist_df['Total'] = dist_df.sum(axis=1)

print("Dataset Distribution:")
print(dist_df.to_string())
print(f"\nTotal images : {dist_df['Total'].sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar chart
dist_df[splits].plot(kind='bar', ax=axes[0],
                     color=['steelblue', 'coral', 'seagreen'],
                     edgecolor='black', width=0.7)
axes[0].set_title('Class distribution across dataset splits', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Number of Images')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(title='Split'); axes[0].grid(axis='y', alpha=0.3)
for container in axes[0].containers:
    axes[0].bar_label(container, fontsize=7, padding=1)

# Pie: total class distribution
axes[1].pie(dist_df['Total'], labels=dist_df.index, autopct='%1.1f%%',
            startangle=140, colors=plt.cm.Set3.colors[:len(dist_df)])
axes[1].set_title('Overall class frequency distribution', fontsize=13, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.png'),
            dpi=300, bbox_inches='tight')
plt.show()

dist_df.to_csv(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.csv'))
print("Saved → dataset_distribution.png, dataset_distribution.csv")


In [ ]:
# ========== NORMALIZED CONFUSION MATRIX (Aggregate across runs) ========== #
cls    = all_runs_results[0]['class_names']
n_cls  = len(cls)
agg_cm = np.zeros((n_cls, n_cls))
for r in all_runs_results:
    agg_cm += confusion_matrix(r['y_true'], r['y_pred']).astype(float)

agg_cm_norm = agg_cm / agg_cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw aggregate
sns.heatmap(agg_cm.astype(int), annot=True, fmt='d',
            xticklabels=display_class_names(cls), yticklabels=display_class_names(cls), cmap='Blues', ax=axes[0])
axes[0].set_title('Pooled confusion matrix counts', fontweight='bold')
format_confusion_matrix_axes(axes[0])

# Normalized (row = true class → shows Recall per class on diagonal)
sns.heatmap(agg_cm_norm, annot=True, fmt='.2%',
            xticklabels=display_class_names(cls), yticklabels=display_class_names(cls), cmap='Blues', ax=axes[1])
axes[1].set_title('Row-normalized recall matrix', fontweight='bold')
format_confusion_matrix_axes(axes[1])

plt.suptitle(f'{experiment_display_name()}: pooled test-set confusion matrix analysis across runs', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'aggregate_confusion_matrix.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print("Saved → aggregate_confusion_matrix.png")

# Print per-class recall from diagonal
print("\nPer-class recall (diagonal of normalized CM):")
for i, cname in enumerate(cls):
    print(f"  {cname:<25} {agg_cm_norm[i, i]:.4f}")


In [ ]:
# ========== MULTI-CLASS ROC CURVES + AUC (One-vs-Rest, Aggregate) ========== #

# Concatenate predictions from all runs
all_y_true   = np.concatenate([r['y_true']     for r in all_runs_results])
all_probs    = np.concatenate([r['pred_probs']  for r in all_runs_results])
cls          = all_runs_results[0]['class_names']
n_cls        = len(cls)
y_bin        = label_binarize(all_y_true, classes=range(n_cls))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
tab_colors = plt.cm.tab10.colors
auc_scores = {}

# --- Per-class ROC ---
fpr_d, tpr_d = {}, {}
for i, cname in enumerate(cls):
    fpr_d[i], tpr_d[i], _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr_d[i], tpr_d[i])
    auc_scores[cname] = roc_auc
    axes[0].plot(fpr_d[i], tpr_d[i], color=tab_colors[i], lw=2,
                 label=f'{cname}  (AUC={roc_auc:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set(xlim=[0, 1], ylim=[0, 1.01],
            xlabel='False positive rate', ylabel='True positive rate',
            title='Per-class one-vs-rest ROC analysis (pooled test predictions)')
axes[0].legend(loc='lower right', fontsize=8); axes[0].grid(alpha=0.3)

# --- Macro-average ROC ---
all_fpr  = np.unique(np.concatenate([fpr_d[i] for i in range(n_cls)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_cls):
    mean_tpr += np.interp(all_fpr, fpr_d[i], tpr_d[i])
mean_tpr  /= n_cls
macro_auc  = auc(all_fpr, mean_tpr)

axes[1].plot(all_fpr, mean_tpr, 'b-', lw=2, label=f'Macro-avg  (AUC={macro_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set(xlim=[0, 1], ylim=[0, 1.01],
            xlabel='False positive rate', ylabel='True positive rate',
            title='Macro-average ROC curve')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.suptitle(f'{experiment_display_name()}: pooled one-vs-rest ROC analysis across runs', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

# --- Print & save ---
print("\nAUC Scores (aggregate):")
print("-" * 40)
for cname, av in auc_scores.items():
    print(f"  {cname:<25} {av:.4f}")
weights      = [np.sum(all_y_true == i) for i in range(n_cls)]
weighted_auc = np.average(list(auc_scores.values()), weights=weights)
print(f"\n  Macro-average AUC   : {macro_auc:.4f}")
print(f"  Weighted-avg AUC    : {weighted_auc:.4f}")

auc_df = pd.DataFrame({'Class': list(auc_scores.keys()), 'AUC': list(auc_scores.values())})
auc_df = pd.concat([auc_df,
                    pd.DataFrame([{'Class': 'Macro-Average', 'AUC': macro_auc},
                                  {'Class': 'Weighted-Average', 'AUC': weighted_auc}])],
                   ignore_index=True)
auc_df.to_csv(os.path.join(BASE_RESULT_DIR, 'auc_scores.csv'), index=False)
print("Saved → roc_curves.png, auc_scores.csv")


In [ ]:
# ========== ADDITIONAL CLASSIFICATION METRICS (Kappa, MCC, Balanced Acc) ========== #
print("="*70)
print("ADDITIONAL METRICS — ALL RUNS")
print("="*70)

extra_rows = []
for r in all_runs_results:
    kappa   = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc     = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    extra_rows.append({
        'Run': r['run'], 'Seed': r['seed'],
        'Accuracy': r['accuracy'],
        'Balanced Accuracy': bal_acc,
        "Cohen's Kappa": kappa,
        'MCC': mcc,
        'F1-Score (w)': r['f1_score'],
    })
    print(f"\nRun {r['run']} (seed={r['seed']}):")
    print(f"  Accuracy          : {r['accuracy']:.4f}")
    print(f"  Balanced Accuracy : {bal_acc:.4f}")
    print(f"  Cohen's Kappa     : {kappa:.4f}")
    print(f"  MCC               : {mcc:.4f}")
    print(f"  F1-Score (w-avg)  : {r['f1_score']:.4f}")

extra_df = pd.DataFrame(extra_rows)
num_cols = ['Accuracy', 'Balanced Accuracy', "Cohen's Kappa", 'MCC', 'F1-Score (w)']
mean_row = {'Run': 'Mean', 'Seed': '—', **{c: extra_df[c].mean() for c in num_cols}}
std_row  = {'Run': 'Std',  'Seed': '—', **{c: extra_df[c].std()  for c in num_cols}}
summary_extra = pd.concat([extra_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

print("\n" + "="*70)
print("SUMMARY TABLE")
print(summary_extra.to_string(index=False))

extra_df.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics.csv'), index=False)
summary_extra.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics_summary.csv'), index=False)

print("\nInterpretation:")
print("  Cohen's Kappa > 0.80 → Almost perfect agreement (Landis & Koch)")
print("  MCC ∈ [−1, 1]; values close to 1 indicate best classification")
print("  Balanced Accuracy corrects for class imbalance")
print("Saved → additional_metrics.csv, additional_metrics_summary.csv")


In [ ]:
# ========== TRAINING CONVERGENCE ANALYSIS ========== #
colors3 = ['steelblue', 'coral', 'seagreen']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Val Accuracy — all runs
for r, c in zip(all_runs_results, colors3):
    axes[0, 0].plot(r['history']['val_accuracy'], color=c,
                    label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 0].set_title('Validation accuracy trajectories across runs', fontweight='bold')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# 2. Val Loss — all runs
for r, c in zip(all_runs_results, colors3):
    axes[0, 1].plot(r['history']['val_loss'], color=c,
                    label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 1].set_title('Validation loss trajectories across runs', fontweight='bold')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

# 3. Best epoch per run
best_epochs   = [np.argmin(r['history']['val_loss']) + 1 for r in all_runs_results]
run_labels    = [f"Run {r['run']}\n(seed={r['seed']})" for r in all_runs_results]
best_val_accs = [r['history']['val_accuracy'][np.argmin(r['history']['val_loss'])]
                 for r in all_runs_results]

bars = axes[1, 0].bar(run_labels, best_epochs, color=colors3[:len(all_runs_results)],
                      edgecolor='black', alpha=0.85)
axes[1, 0].set_title('Best epoch by validation loss per run', fontweight='bold')
axes[1, 0].set_ylabel('Epoch'); axes[1, 0].grid(axis='y', alpha=0.3)
for bar, ep in zip(bars, best_epochs):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    str(ep), ha='center', fontweight='bold')

# 4. Train-Val accuracy gap (overfitting indicator)
for r, c in zip(all_runs_results, colors3):
    gap = np.array(r['history']['accuracy']) - np.array(r['history']['val_accuracy'])
    axes[1, 1].plot(gap, color=c, label=f"Run {r['run']}", lw=2)
axes[1, 1].axhline(0, color='black', linestyle='--', lw=1)
axes[1, 1].set_title('Generalization gap (training - validation accuracy)', fontweight='bold')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Train Acc − Val Acc')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.suptitle(f'{experiment_display_name()}: training convergence analysis across runs', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, 'convergence_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Best epochs    :", {f"Run {r['run']}": be for r, be in zip(all_runs_results, best_epochs)})
print("Val acc @ best :", {f"Run {r['run']}": f"{a:.4f}" for r, a in zip(all_runs_results, best_val_accs)})
print("Saved → convergence_analysis.png")


In [ ]:
# ========== SELECT RUN FOR SINGLE-RUN ANALYSIS ========== #
# Change SELECTED_RUN to any value in [1, len(RANDOM_SEEDS[:N_RUNS])] to analyse a different experiment.
SELECTED_RUN = len(RANDOM_SEEDS)   # default: last run

run_data    = all_runs_results[SELECTED_RUN - 1]
RESULT_DIR  = run_data['result_dir']
y_true      = run_data['y_true']
y_pred      = run_data['y_pred']
pred_probs  = run_data['pred_probs']
class_names = run_data['class_names']
test_acc    = run_data['accuracy']

# Dataset stats (for summary report)
n_train     = run_data['n_train']
n_val       = run_data['n_val']
n_test      = run_data['n_test']

# File paths for misclassification / Grad-CAM analysis
test_filenames = run_data['test_filenames']   # list of "classname/filename.jpg"
test_dir       = os.path.join(DATA_DIR, 'test')

# Wrap history dict so history.history['key'] syntax works in downstream cells
history = SimpleNamespace(history=run_data['history'])

# Reload best model (needed for Grad-CAM, t-SNE, inference speed, etc.)
custom_objects = {
    "CastToFloat32": CastToFloat32,
    "TopKPooling2D": TopKPooling2D,
    "DiversityRegularizer": DiversityRegularizer,
    "GatedLogitFusion": GatedLogitFusion,
    "ClassBalancedFocalLoss": ClassBalancedFocalLoss,
}
model = load_model(
    os.path.join(RESULT_DIR, 'best_model.keras'),
    custom_objects=custom_objects,
    compile=False,
)

# Rebuild test tf.data pipeline for inference-speed measurement and t-SNE
_, _, test_ds, _ = create_tf_datasets(
    DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=run_data['seed'])

print(f"Analysing Run {SELECTED_RUN}  (seed={run_data['seed']})")
print(f"  Classes  : {class_names}")
print(f"  Train / Val / Test : {n_train} / {n_val} / {n_test}")
print(f"  Accuracy : {run_data['accuracy']:.4f}")
print(f"  F1-Score : {run_data['f1_score']:.4f}")
print(f"  Dir      : {RESULT_DIR}")


In [ ]:
# ========== GRAD-CAM VISUALIZATION ========== #

def make_gradcam_heatmap(img_array, grad_model, pred_index=None):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap(orig_img_array, heatmap, alpha=0.4):
    heatmap_uint8 = np.uint8(255 * heatmap)
    jet_colors    = plt.cm.jet(np.arange(256))[:, :3]
    jet_heatmap   = jet_colors[heatmap_uint8]
    jet_heatmap   = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap   = jet_heatmap.resize((orig_img_array.shape[1], orig_img_array.shape[0]))
    jet_heatmap   = img_to_array(jet_heatmap)
    superimposed  = jet_heatmap * alpha + orig_img_array
    return np.clip(superimposed / superimposed.max(), 0, 1)

# Locate Grad-CAM target layer (robust to nested models)
def _pick_gradcam_target(model):
    def _get_4d_tensor(layer):
        out = layer.output
        outs = out if isinstance(out, (list, tuple)) else [out]
        for t in reversed(outs):
            shp = getattr(t, 'shape', None)
            if shp is not None and len(shp) == 4:
                return t
        return None

    # Prefer the shared GDE feature map, then the EfficientNetB4 semantic map.
    for name in ['shared_reduce_bn', 'shared_reduce_conv', 'top_activation', 'efficientnetb4']:
        try:
            layer = model.get_layer(name)
            tensor = _get_4d_tensor(layer)
            if tensor is not None:
                return layer.name, tensor
        except Exception:
            pass

    # Fallback: last top-level layer that has a 4D output tensor
    for layer in reversed(model.layers):
        tensor = _get_4d_tensor(layer)
        if tensor is not None:
            return layer.name, tensor

    raise ValueError('No suitable 4D feature-map layer found for Grad-CAM.')

last_conv_name, gradcam_tensor = _pick_gradcam_target(model)
print(f"Grad-CAM target layer: {last_conv_name}")

grad_model = Model(
    inputs=model.inputs,
    outputs=[gradcam_tensor, model.output],
)

# test_dir and test_filenames are set in the Select Run cell
n_cls    = len(class_names)
fig, axes = plt.subplots(n_cls, 3, figsize=(12, 4 * n_cls))
if n_cls == 1:
    axes = axes[np.newaxis, :]

for ci, cname in enumerate(class_names):
    # Pick first correctly classified sample for this class
    ok_idx = np.where((y_true == ci) & (y_pred == ci))[0]
    idx    = ok_idx[0] if len(ok_idx) > 0 else np.where(y_true == ci)[0][0]
    fpath  = os.path.join(test_dir, test_filenames[idx])   # test_filenames from Select Run

    # Preprocess
    img_orig = load_img(fpath, target_size=INPUT_SHAPE[:2])
    img_arr  = img_to_array(img_orig)
    img_proc = efficientnet_preprocess(np.expand_dims(img_arr.copy(), 0))

    # Grad-CAM
    heatmap  = make_gradcam_heatmap(img_proc, grad_model, pred_index=ci)
    overlay  = overlay_heatmap(img_arr, heatmap)

    # Plot
    axes[ci, 0].imshow(img_orig); axes[ci, 0].axis('off')
    axes[ci, 0].set_title(f'Input image\nTrue class: {format_class_label(cname)}', fontsize=9)
    axes[ci, 1].imshow(heatmap, cmap='jet'); axes[ci, 1].axis('off')
    axes[ci, 1].set_title('Grad-CAM activation map', fontsize=9)
    axes[ci, 2].imshow(overlay); axes[ci, 2].axis('off')
    conf = pred_probs[idx][y_pred[idx]]
    axes[ci, 2].set_title(f'Activation overlay\nPred. confidence={conf:.2%}', fontsize=9)

plt.suptitle(f'{experiment_display_name()}: Grad-CAM activation visualizations (run {SELECTED_RUN})',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(RESULT_DIR, 'gradcam_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → gradcam_visualization.png")


In [ ]:
# ========== GRAD-CAM++ VISUALIZATION ========== #
# Grad-CAM++ (Chattopadhay et al., 2018) provides more complete localisation
# than standard Grad-CAM by using second- and third-order gradients to weight
# the feature maps — especially effective for multi-object / multi-region cases.

import matplotlib.cm as cm_lib

def gradcam_pp(model, img_array, layer_name, class_idx=None):
    """
    Grad-CAM++ activation map. Works with mixed-precision models (float16 layers).
    """
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output],
    )
    img_tensor = tf.cast(img_array, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        conv_out_orig, preds = grad_model(img_tensor, training=False)
        tape.watch(conv_out_orig)
        preds_f32 = tf.cast(preds, tf.float32)
        if class_idx is None:
            class_idx = tf.argmax(preds_f32[0])
        loss = preds_f32[:, class_idx]

    grads    = tape.gradient(loss, conv_out_orig)
    grads    = tf.cast(grads, tf.float32)
    conv_out = tf.cast(conv_out_orig, tf.float32)

    grads_sq  = grads ** 2
    grads_cu  = grads ** 3
    alpha_num = grads_sq
    alpha_den = 2 * grads_sq + conv_out * grads_cu
    alpha_den = tf.where(alpha_den == 0, tf.ones_like(alpha_den), alpha_den)
    alpha     = alpha_num / alpha_den
    relu_grad = tf.nn.relu(loss * grads)
    weights   = tf.reduce_mean(alpha * relu_grad, axis=(1, 2))[0]
    cam       = tf.reduce_sum(conv_out[0] * weights, axis=-1).numpy()
    cam       = np.maximum(cam, 0)
    if cam.max() > 0:
        cam = cam / cam.max()
    return cam


# Find target layer for Grad-CAM++
gradcam_pp_layer = last_conv_name  # reuse from Grad-CAM cell above
print(f"Grad-CAM++ target layer: {gradcam_pp_layer}")

h, w = INPUT_SHAPE[:2]

# Sample images: 2 per class (mix of correct and incorrect if available)
sample_imgs, sample_paths, sample_labels, sample_indices = [], [], [], []
for ci, cname in enumerate(class_names):
    # Try to get 1 correct + 1 wrong
    correct_idx = np.where((y_true == ci) & (y_pred == ci))[0]
    wrong_idx_c = np.where((y_true == ci) & (y_pred != ci))[0]
    chosen = []
    if len(correct_idx) > 0:
        chosen.append(correct_idx[0])
    if len(wrong_idx_c) > 0:
        chosen.append(wrong_idx_c[0])
    if len(chosen) < 2 and len(correct_idx) > 1:
        chosen.append(correct_idx[1])
    if len(chosen) < 2:
        all_cls = np.where(y_true == ci)[0]
        for a in all_cls:
            if a not in chosen:
                chosen.append(a)
                break
    for idx in chosen[:2]:
        fpath = os.path.join(test_dir, test_filenames[idx])
        img_orig = load_img(fpath, target_size=(h, w))
        img_arr = img_to_array(img_orig)
        sample_imgs.append(img_arr)
        sample_paths.append(fpath)
        sample_labels.append(cname)
        sample_indices.append(idx)

n_samples = len(sample_imgs)
fig, axes = plt.subplots(n_samples, 3, figsize=(11, n_samples * 3.2))
fig.suptitle(f'{experiment_display_name()}: Grad-CAM++ activation visualizations\n(run {SELECTED_RUN}, seed={run_data["seed"]})',
             fontsize=11, fontweight='bold')

for row in range(n_samples):
    orig = sample_imgs[row]
    idx = sample_indices[row]
    img_proc = efficientnet_preprocess(np.expand_dims(orig.copy(), 0))

    cam = gradcam_pp(model, img_proc, gradcam_pp_layer)
    cam_up = tf.image.resize(cam[..., np.newaxis], [h, w]).numpy()[:, :, 0]
    hmap = (cm_lib.jet(cam_up)[:, :, :3] * 255).astype(np.uint8)
    orig_uint8 = np.clip(orig, 0, 255).astype(np.uint8)
    overlay = (0.55 * orig_uint8 + 0.45 * hmap).astype(np.uint8)

    pred_idx = int(y_pred[idx])
    pred_name = class_names[pred_idx]
    correct = sample_labels[row] == pred_name
    conf = float(pred_probs[idx][pred_idx])

    axes[row, 0].imshow(orig_uint8)
    axes[row, 0].set_title(f'Ground truth: {sample_labels[row]}', fontsize=8)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(cam_up, cmap='jet')
    axes[row, 1].set_title('Grad-CAM++ activation map', fontsize=8)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(
        f'Pred: {pred_name} ({conf:.2f})', fontsize=8,
        color='green' if correct else 'red')
    axes[row, 2].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(RESULT_DIR, 'gradcam_pp.png'), dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved Grad-CAM++ visualization -> {RESULT_DIR}/gradcam_pp.png")

In [ ]:
# ========== EVIDENCE MAP AND GATE VISUALIZATION ========== #
# This cell runs for GDE-Net variants that contain coverage_map and/or defect_map.
# It is skipped automatically for V0/V1.

evidence_layer_names = [
    name for name in ["coverage_map", "defect_map"]
    if name in [layer.name for layer in model.layers]
]
gate_layer_names = [
    name for name in ["coverage_gate", "defect_gate", "evidence_gate"]
    if name in [layer.name for layer in model.layers]
]

if len(evidence_layer_names) == 0:
    print("Evidence maps skipped: this ablation variant has no coverage/defect evidence maps.")
else:
    import matplotlib.cm as cm_lib

    def _normalize_map(m):
        m = np.asarray(m, dtype=np.float32)
        return (m - m.min()) / (m.max() - m.min() + 1e-8)

    def _overlay_map(orig_uint8, evidence_map, alpha=0.42):
        h, w = orig_uint8.shape[:2]
        resized = tf.image.resize(evidence_map[..., np.newaxis], [h, w]).numpy()[:, :, 0]
        resized = _normalize_map(resized)
        heat = (cm_lib.jet(resized)[:, :, :3] * 255).astype(np.uint8)
        return (orig_uint8 * (1.0 - alpha) + heat * alpha).astype(np.uint8), resized

    if "sample_paths" not in globals() or len(sample_paths) == 0:
        sample_paths, sample_labels, sample_indices = [], [], []
        for ci, cname in enumerate(class_names):
            idxs = np.where(y_true == ci)[0]
            if len(idxs) == 0:
                continue
            idx = int(idxs[0])
            sample_paths.append(os.path.join(test_dir, test_filenames[idx]))
            sample_labels.append(cname)
            sample_indices.append(idx)

    probe_outputs = [model.get_layer(name).output for name in evidence_layer_names + gate_layer_names]
    evidence_probe = Model(inputs=model.input, outputs=probe_outputs)

    n_samples = min(len(sample_paths), max(3, len(class_names)))
    n_cols = 1 + len(evidence_layer_names)
    fig, axes = plt.subplots(n_samples, n_cols, figsize=(4 * n_cols, 3.3 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    for row in range(n_samples):
        img_orig = load_img(sample_paths[row], target_size=INPUT_SHAPE[:2])
        img_arr = img_to_array(img_orig)
        orig_uint8 = np.clip(img_arr, 0, 255).astype(np.uint8)
        img_proc = efficientnet_preprocess(np.expand_dims(img_arr.copy(), 0))

        outputs = evidence_probe(img_proc, training=False)
        if not isinstance(outputs, (list, tuple)):
            outputs = [outputs]
        map_outputs = outputs[:len(evidence_layer_names)]

        idx = sample_indices[row]
        pred_idx = int(y_pred[idx])
        conf = float(pred_probs[idx][pred_idx])
        axes[row, 0].imshow(orig_uint8)
        axes[row, 0].axis("off")
        axes[row, 0].set_title(
            f"True: {sample_labels[row]}\nPred: {class_names[pred_idx]} ({conf:.2f})",
            fontsize=8,
            color="green" if y_true[idx] == pred_idx else "red",
        )

        for col, (layer_name, map_tensor) in enumerate(zip(evidence_layer_names, map_outputs), start=1):
            evidence_map = np.asarray(map_tensor[0, :, :, 0], dtype=np.float32)
            overlay, resized = _overlay_map(orig_uint8, evidence_map)
            axes[row, col].imshow(overlay)
            axes[row, col].axis("off")
            axes[row, col].set_title(f"{layer_name}\nmean={resized.mean():.3f}", fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(os.path.join(RESULT_DIR, "evidence_maps.png"), dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved evidence map visualization -> {RESULT_DIR}/evidence_maps.png")

    # Aggregate map/gate statistics by true class on the selected test set.
    stat_rows = []
    accum = {
        cname: {
            "coverage_activation": [],
            "defect_activation": [],
            "alpha_cov": [],
            "alpha_def": [],
        }
        for cname in class_names
    }

    for batch_x, batch_y in test_ds:
        outputs = evidence_probe(batch_x, training=False)
        if not isinstance(outputs, (list, tuple)):
            outputs = [outputs]
        map_outputs = outputs[:len(evidence_layer_names)]
        gate_outputs = outputs[len(evidence_layer_names):]
        y_batch = np.argmax(batch_y.numpy(), axis=1)

        for bi, yi in enumerate(y_batch):
            cname = class_names[int(yi)]
            for layer_name, map_tensor in zip(evidence_layer_names, map_outputs):
                val = float(np.mean(np.asarray(map_tensor[bi], dtype=np.float32)))
                if layer_name == "coverage_map":
                    accum[cname]["coverage_activation"].append(val)
                elif layer_name == "defect_map":
                    accum[cname]["defect_activation"].append(val)

            for gate_name, gate_tensor in zip(gate_layer_names, gate_outputs):
                gate_val = np.asarray(gate_tensor[bi], dtype=np.float32).reshape(-1)
                if gate_name == "coverage_gate":
                    accum[cname]["alpha_cov"].append(float(gate_val[0]))
                elif gate_name == "defect_gate":
                    accum[cname]["alpha_def"].append(float(gate_val[0]))
                elif gate_name == "evidence_gate":
                    if len(gate_val) > 0:
                        accum[cname]["alpha_cov"].append(float(gate_val[0]))
                    if len(gate_val) > 1:
                        accum[cname]["alpha_def"].append(float(gate_val[1]))

    for cname, vals in accum.items():
        row = {"class": cname}
        for metric_name, metric_vals in vals.items():
            row[metric_name] = float(np.mean(metric_vals)) if len(metric_vals) else np.nan
        stat_rows.append(row)

    evidence_stats_df = pd.DataFrame(stat_rows)
    evidence_stats_path = os.path.join(RESULT_DIR, "evidence_activation_by_class.csv")
    evidence_stats_df.to_csv(evidence_stats_path, index=False)
    print("\nEvidence/gate statistics by class:")
    print(evidence_stats_df.to_string(index=False))
    print(f"Saved -> {evidence_stats_path}")


In [ ]:
# ========== GRAD-CAM++ ON MISCLASSIFIED IMAGES ========== #
# Highlights WHERE the model looked when making wrong predictions.
# Helps diagnose whether errors are due to:
#   - Looking at background / irrelevant regions
#   - Confusing textures between similar disease classes
#   - Low-quality / ambiguous input images

wrong_indices_pp = np.where(y_true != y_pred)[0]
# Sort by confidence (highest confidence wrong = most informative)
wrong_confs = [float(pred_probs[i][y_pred[i]]) for i in wrong_indices_pp]
sorted_wrong = [wrong_indices_pp[i] for i in np.argsort(-np.array(wrong_confs))]

TOP_WRONG = min(10, len(sorted_wrong))
print(f"Generating Grad-CAM++ for top-{TOP_WRONG} high-confidence misclassifications...")

if TOP_WRONG > 0:
    n_cols = 3
    fig, axes = plt.subplots(TOP_WRONG, n_cols, figsize=(11, TOP_WRONG * 3.2))
    if TOP_WRONG == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(f'{experiment_display_name()}: Grad-CAM++ activation analysis for high-confidence misclassifications\n'
                 f'Run {SELECTED_RUN}',
                 fontsize=11, fontweight='bold', color='crimson')

    for row, idx in enumerate(sorted_wrong[:TOP_WRONG]):
        fpath = os.path.join(test_dir, test_filenames[idx])
        img_orig = load_img(fpath, target_size=(h, w))
        img_arr = img_to_array(img_orig)
        orig_uint8 = np.clip(img_arr, 0, 255).astype(np.uint8)
        img_proc = efficientnet_preprocess(np.expand_dims(img_arr.copy(), 0))

        # Grad-CAM++ on the predicted (wrong) class
        cam = gradcam_pp(model, img_proc, gradcam_pp_layer, class_idx=int(y_pred[idx]))
        cam_up = tf.image.resize(cam[..., np.newaxis], [h, w]).numpy()[:, :, 0]
        hmap = (cm_lib.jet(cam_up)[:, :, :3] * 255).astype(np.uint8)
        overlay = (0.55 * orig_uint8 + 0.45 * hmap).astype(np.uint8)

        true_name = class_names[int(y_true[idx])]
        pred_name = class_names[int(y_pred[idx])]
        true_label = format_class_label(true_name)
        pred_label = format_class_label(pred_name)
        pred_conf = float(pred_probs[idx][y_pred[idx]])
        true_conf = float(pred_probs[idx][y_true[idx]])

        axes[row, 0].imshow(orig_uint8)
        axes[row, 0].set_title(f'Ground truth: {true_label} (p={true_conf:.2f})', fontsize=7)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(cam_up, cmap='jet')
        axes[row, 1].set_title('Grad-CAM++ activation for predicted class', fontsize=7)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(f'Predicted: {pred_label} (p={pred_conf:.2f})',
                                fontsize=7, color='red', fontweight='bold')
        axes[row, 2].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(os.path.join(RESULT_DIR, 'gradcam_pp_misclassified.png'),
                dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved Grad-CAM++ misclassified -> {RESULT_DIR}/gradcam_pp_misclassified.png")
else:
    print("No misclassifications found — perfect accuracy on test set!")

In [ ]:
# ========== t-SNE FEATURE EMBEDDING VISUALIZATION ========== #

# Use penultimate representation from V1 GAP+GMP head for embedding analysis.
feature_layer_name = 'head_dense'
feature_extractor = Model(inputs=model.input, outputs=model.get_layer(feature_layer_name).output)

print("Extracting features from test set...")
features = feature_extractor.predict(test_ds, verbose=1)

print("Computing t-SNE embedding...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
features_2d = tsne.fit_transform(features)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
palette = plt.cm.tab10.colors

# Left: color by true class
for ci, cname in enumerate(class_names):
    mask = y_true == ci
    axes[0].scatter(features_2d[mask, 0], features_2d[mask, 1],
                    c=[palette[ci % len(palette)]], label=cname, alpha=0.7, s=20)
axes[0].set_title(f't-SNE feature embedding by ground-truth class (run {SELECTED_RUN})', fontweight='bold')
axes[0].legend(markerscale=2, fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].set_xlabel('t-SNE dimension 1')
axes[0].set_ylabel('t-SNE dimension 2')

# Right: correct vs misclassified
correct_mask = y_true == y_pred
axes[1].scatter(features_2d[correct_mask, 0], features_2d[correct_mask, 1],
                c='steelblue', label='Correct', alpha=0.7, s=20)
axes[1].scatter(features_2d[~correct_mask, 0], features_2d[~correct_mask, 1],
                c='red', label='Misclassified', alpha=0.8, s=24, marker='x')
axes[1].set_title(f't-SNE feature embedding by prediction outcome (run {SELECTED_RUN})', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_xlabel('t-SNE dimension 1')
axes[1].set_ylabel('t-SNE dimension 2')

plt.suptitle(f'{experiment_display_name()}: t-SNE feature embedding (run {SELECTED_RUN})',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(RESULT_DIR, 'tsne_features.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved -> tsne_features.png")


In [ ]:
# ========== ERROR ANALYSIS ========== #
print("="*70)
print("ERROR ANALYSIS")
print("="*70)

# --- Misclassification pairs ---
confusion_counts = defaultdict(int)
for yt, yp in zip(y_true, y_pred):
    if yt != yp:
        confusion_counts[(class_names[yt], class_names[yp])] += 1

print(f"\nTotal misclassifications: {sum(confusion_counts.values())} / {len(y_true)}")
print(f"Overall accuracy: {np.mean(y_true == y_pred):.4f}\n")
print("Most common confused pairs (True → Predicted):")
for (tc, pc), cnt in sorted(confusion_counts.items(), key=lambda x: -x[1])[:10]:
    pct = cnt / np.sum(y_true == class_names.index(tc)) * 100
    print(f"  {tc:<22} → {pc:<22}  {cnt:3d} samples  ({pct:.1f}% of class)")

# --- Confidence distribution ---
correct_mask = y_true == y_pred
correct_conf = pred_probs[correct_mask][np.arange(correct_mask.sum()), y_pred[correct_mask]]
wrong_conf   = pred_probs[~correct_mask][np.arange((~correct_mask).sum()), y_pred[~correct_mask]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(correct_conf, bins=20, color='steelblue', alpha=0.7, edgecolor='black',
             label=f'Correct (n={len(correct_conf)})')
axes[0].hist(wrong_conf, bins=20, color='salmon', alpha=0.7, edgecolor='black',
             label=f'Wrong (n={len(wrong_conf)})')
axes[0].axvline(0.5, color='black', linestyle='--', lw=1)
axes[0].set_title('Prediction confidence distribution', fontweight='bold')
axes[0].set_xlabel('Confidence (softmax probability)')
axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].grid(alpha=0.3)

# --- Per-class accuracy bar ---
class_acc = [(cn, np.mean(y_pred[y_true == ci] == ci), (y_true == ci).sum())
             for ci, cn in enumerate(class_names)]
class_acc.sort(key=lambda x: x[1])
names, accs, counts = zip(*class_acc)
colors_bar = plt.cm.RdYlGn(np.array(accs))
bars = axes[1].barh(names, accs, color=colors_bar, edgecolor='black', height=0.6)
axes[1].set_title('Class-wise test accuracy (sorted)', fontweight='bold')
axes[1].set_xlabel('Accuracy'); axes[1].set_xlim([0, 1.15])
axes[1].grid(axis='x', alpha=0.3)
for bar, acc, n in zip(bars, accs, counts):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{acc:.3f}  (n={n})', va='center', fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(RESULT_DIR, 'error_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

# --- High-confidence wrong predictions ---
wrong_idx = np.where(y_true != y_pred)[0]
wrong_conf_vals = pred_probs[wrong_idx][np.arange(len(wrong_idx)), y_pred[wrong_idx]]
top_wrong = wrong_idx[np.argsort(-wrong_conf_vals)[:10]]
print("\nTop-10 high-confidence incorrect predictions:")
for i, idx in enumerate(top_wrong):
    fname = test_filenames[idx] if idx < len(test_filenames) else str(idx)
    print(f"  {i+1:2d}. True={class_names[y_true[idx]]:<20}  "
          f"Pred={class_names[y_pred[idx]]:<20}  "
          f"Conf={pred_probs[idx][y_pred[idx]]:.4f}  ({fname})")
print("\nSaved → error_analysis.png")


In [ ]:
# ========== TOP-5 MISCLASSIFIED IMAGES PER CONFUSION PAIR ==========
import gc

TOP_N         = 5
GRADCAM_ALPHA = 0.45

# test_dir and test_filenames are set in the Select Run cell
wrong_indices = np.where(y_true != y_pred)[0]

# ---- Build Grad-CAM model ----
# Reuse helper from previous Grad-CAM cell (define fallback if cell order changes)
if '_pick_gradcam_target' not in globals():
    def _pick_gradcam_target(model):
        def _get_4d_tensor(layer):
            out = layer.output
            outs = out if isinstance(out, (list, tuple)) else [out]
            for t in reversed(outs):
                shp = getattr(t, 'shape', None)
                if shp is not None and len(shp) == 4:
                    return t
            return None
        for name in ['se_semantic', 'se_local', 'efficientnetb4_multiscale']:
            try:
                layer = model.get_layer(name)
                tensor = _get_4d_tensor(layer)
                if tensor is not None:
                    return layer.name, tensor
            except Exception:
                pass
        for layer in reversed(model.layers):
            tensor = _get_4d_tensor(layer)
            if tensor is not None:
                return layer.name, tensor
        raise ValueError('No suitable 4D feature-map layer found for Grad-CAM.')

last_conv_name, gradcam_tensor = _pick_gradcam_target(model)
grad_model_vis = tf.keras.Model(
    inputs  = model.inputs,
    outputs = [gradcam_tensor, model.output]
)

def _gradcam(img_path, target_class):
    # Returns (orig_float, heatmap, overlay, confidence).
    # Returns (orig_float, None, orig_float, 0.0) gracefully on any error.
    try:
        img_orig = load_img(img_path, target_size=INPUT_SHAPE[:2])
        img_arr  = img_to_array(img_orig)
        img_proc = efficientnet_preprocess(
            tf.cast(np.expand_dims(img_arr.copy(), 0), tf.float32))

        with tf.GradientTape() as tape:
            conv_out, preds = grad_model_vis(img_proc, training=False)
            class_score     = preds[:, target_class]
        grads   = tape.gradient(class_score, conv_out)
        pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
        heatmap = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
        heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
        heatmap = heatmap.numpy()

        h_uint8    = np.uint8(255 * heatmap)
        jet_colors = plt.cm.jet(np.arange(256))[:, :3]
        jet_hm_img = tf.keras.preprocessing.image.array_to_img(jet_colors[h_uint8])
        jet_hm_img = jet_hm_img.resize((img_arr.shape[1], img_arr.shape[0]))
        jet_hm_arr = img_to_array(jet_hm_img)
        overlay    = jet_hm_arr * GRADCAM_ALPHA + img_arr
        overlay    = np.clip(overlay / overlay.max(), 0, 1)

        conf       = float(preds.numpy()[0, target_class])
        orig_float = np.clip(img_arr / 255.0, 0, 1)
        return orig_float, heatmap, overlay, conf

    except Exception as e:
        print(f'  [WARN] Grad-CAM failed for {img_path}: {type(e).__name__}: {str(e)[:80]}')
        try:
            orig_float = np.clip(img_to_array(
                load_img(img_path, target_size=INPUT_SHAPE[:2])) / 255.0, 0, 1)
        except Exception:
            orig_float = np.zeros((*INPUT_SHAPE[:2], 3))
        return orig_float, None, orig_float, 0.0


# ---- Group wrong samples by (true, pred) pair ----
pair_dict = defaultdict(list)
for idx in wrong_indices:
    pair = (int(y_true[idx]), int(y_pred[idx]))
    pair_dict[pair].append((idx, float(pred_probs[idx][y_pred[idx]])))

sorted_pairs = sorted(pair_dict.items(), key=lambda x: -len(x[1]))
print(f'Found {len(sorted_pairs)} unique (True -> Predicted) confusion pairs')
print(f'Generating Top-{TOP_N} misclassified image panels...\n')

report_rows = []

for (ti, pi), samples in sorted_pairs:
    true_name   = class_names[ti]
    pred_name   = class_names[pi]
    true_label  = format_class_label(true_name)
    pred_label  = format_class_label(pred_name)
    n_total     = len(samples)
    top_samples = sorted(samples, key=lambda x: -x[1])[:TOP_N]
    n_show      = len(top_samples)

    fig, axes = plt.subplots(n_show, 3, figsize=(12, max(3.5, n_show * 3.5)))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(
        f'Misclassified test images: {true_label} predicted as {pred_label} (n={n_total})',
        fontsize=11, fontweight='bold', y=0.98
    )
    for col, title in enumerate(['Original image', 'Grad-CAM activation map', 'Overlay']):
        axes[0, col].set_title(title, fontsize=10, fontweight='bold', pad=6)

    for row, (idx, _) in enumerate(top_samples):
        fpath     = os.path.join(test_dir, test_filenames[idx])
        true_conf = float(pred_probs[idx][ti])
        pred_conf = float(pred_probs[idx][pi])

        orig, heatmap, overlay, _ = _gradcam(fpath, pi)

        # Col 0: original
        axes[row, 0].imshow(orig); axes[row, 0].axis('off')
        axes[row, 0].set_ylabel(f'Sample {row+1}', fontsize=8,
                                 rotation=0, labelpad=40, va='center')
        axes[row, 0].text(0.02, 0.02, f'Ground truth: {true_label}\n(p={true_conf:.2%})',
            transform=axes[row, 0].transAxes, fontsize=8, color='lime',
            fontweight='bold', bbox=dict(facecolor='black', alpha=0.55, pad=2))

        # Col 1: heatmap or placeholder
        if heatmap is not None:
            axes[row, 1].imshow(heatmap, cmap='jet')
            axes[row, 1].text(0.02, 0.02, f'Layer:\n{last_conv_name}',
                transform=axes[row, 1].transAxes, fontsize=7, color='white',
                bbox=dict(facecolor='black', alpha=0.55, pad=2))
        else:
            axes[row, 1].imshow(orig)
            axes[row, 1].text(0.02, 0.02, 'Grad-CAM\nunavailable\n(OOM)',
                transform=axes[row, 1].transAxes, fontsize=8, color='red',
                fontweight='bold', bbox=dict(facecolor='black', alpha=0.60, pad=2))
        axes[row, 1].axis('off')

        # Col 2: overlay
        top3_idx = np.argsort(-pred_probs[idx])[:3]
        top3_str = '\n'.join(
            [f'  {format_class_label(class_names[k])}: {pred_probs[idx][k]:.2%}' for k in top3_idx])
        axes[row, 2].imshow(overlay); axes[row, 2].axis('off')
        axes[row, 2].text(0.02, 0.02,
            f'Predicted: {pred_label}\n(p={pred_conf:.2%})\n\nTop-3:\n{top3_str}',
            transform=axes[row, 2].transAxes, fontsize=7.5,
            color='yellow', fontweight='bold',
            bbox=dict(facecolor='black', alpha=0.60, pad=3))

        report_rows.append({
            'true_class': true_label,
            'pred_class': pred_label,
            'true_source_label': true_name,
            'pred_source_label': pred_name,
            'sample':        row + 1,
            'filename':      test_filenames[idx],
            'true_conf':     round(true_conf, 4),
            'pred_conf':     round(pred_conf, 4),
            'gradcam_ok':    heatmap is not None,
            'total_in_pair': n_total,
        })

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    safe_fname = f'top{TOP_N}_wrong_{true_name}_as_{pred_name}.png'
    plt.savefig(os.path.join(RESULT_DIR, safe_fname), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'  [{true_label} -> {pred_label}]  {n_total} errors - saved {safe_fname}')
    gc.collect()

# ---- Save CSV ----
wrong_report_df = pd.DataFrame(report_rows)
csv_path = os.path.join(RESULT_DIR, 'top5_misclassified_report.csv')
wrong_report_df.to_csv(csv_path, index=False)
print(f'\nCSV saved ({len(wrong_report_df)} rows) -> top5_misclassified_report.csv')
print('Done.')


In [ ]:
# ========== PREPARE PATHS ========== #
# test_filenames = list of "classname/filename.jpg" (set in Select Run cell)
filepaths = [os.path.join(test_dir, f) for f in test_filenames]


In [ ]:
# ========== FIND CONFUSION TYPES ========== #
from collections import defaultdict

confusion_dict = defaultdict(list)

for i in range(len(y_true)):
    if y_true[i] != y_pred[i]:
        key = (class_names[y_true[i]], class_names[y_pred[i]])
        confidence = pred_probs[i][y_pred[i]]
        confusion_dict[key].append((filepaths[i], confidence, i))

print("Total confusion types:", len(confusion_dict))


In [ ]:
# ========== SELECT REPRESENTATIVE IMAGES ========== #
import shutil

analysis_dir = os.path.join(RESULT_DIR, "qualitative_analysis")
os.makedirs(analysis_dir, exist_ok=True)

summary_lines = []

for (true_label, pred_label), samples in confusion_dict.items():

    # sort theo độ tự tin giảm dần
    samples_sorted = sorted(samples, key=lambda x: x[1], reverse=True)

    selected = samples_sorted[:10]  # lấy 2 ảnh

    pair_folder = os.path.join(analysis_dir, f"{true_label}_as_{pred_label}")
    os.makedirs(pair_folder, exist_ok=True)

    summary_lines.append(f"\n=== {true_label} → {pred_label} ===")

    for idx, (img_path, conf, i) in enumerate(selected):
        new_name = f"sample_{idx+1}_conf_{conf:.3f}.jpg"
        dst = os.path.join(pair_folder, new_name)
        shutil.copy(img_path, dst)

        summary_lines.append(f"{new_name} | confidence={conf:.3f}")


In [ ]:
with open(os.path.join(analysis_dir, "analysis_notes.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("Saved qualitative analysis samples")


In [ ]:
qual_dir = os.path.join(RESULT_DIR, "qualitative_analysis")
zip_out  = "/kaggle/working/qualitative_analysis.zip"
shutil.make_archive(zip_out.replace(".zip", ""), 'zip', qual_dir)
print(f"Qualitative analysis zipped → {zip_out}")


---
## Section 4 — Single-Run Model Analysis & Full Report

Detailed model diagnostics: architecture, size, inference speed, Top-K accuracy, per-class metrics, and full summary report. All cells use the run selected above (`SELECTED_RUN`).


In [ ]:
# ========== MODEL SUMMARY & PARAMETERS ========== #
model.summary()

# Count parameters
total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_params = total_params - trainable_params

print("\n" + "="*60)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Non-trainable params: {non_trainable_params:,}")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "model_summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))
    f.write("\n" + "="*60 + "\n")
    f.write(f"Total params: {total_params:,}\n")
    f.write(f"Trainable params: {trainable_params:,}\n")
    f.write(f"Non-trainable params: {non_trainable_params:,}\n")
    f.write("="*60 + "\n")

print("✅ Model summary saved")


In [ ]:
# ========== MODEL SIZE ========== #
import tempfile

# Save model temporarily to get size
temp_model_path = os.path.join(tempfile.gettempdir(), "temp_model.keras")
model.save(temp_model_path)
model_size_bytes = os.path.getsize(temp_model_path)
model_size_mb = model_size_bytes / (1024 * 1024)

print("="*60)
print(f"Model size: {model_size_mb:.2f} MB ({model_size_bytes:,} bytes)")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "model_size.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write(f"Model size: {model_size_mb:.2f} MB ({model_size_bytes:,} bytes)\n")
    f.write("="*60 + "\n")

# Clean up
os.remove(temp_model_path)
print("✅ Model size saved")


In [ ]:
# ========== INFERENCE SPEED ========== #
# Measures latency on the test set using model.predict (batch-level).
# tf.keras.backend.clear_session() was called at end of each training run,
# so GPU memory is freed. We use small batch size here to avoid OOM.

SPEED_BATCH = 8    # small batch — avoids OOM after multi-run training
MAX_BATCHES = 10

print("Measuring inference speed...")

# Rebuild a small-batch version of test_ds just for timing
speed_ds = (
    tf.data.Dataset.from_tensor_slices((
        [os.path.join(DATA_DIR, "test", f) for f in test_filenames],
        y_true,
    ))
    .map(lambda p, l: (
        efficientnet_preprocess(
            tf.cast(tf.image.resize(
                tf.image.decode_jpeg(tf.io.read_file(p), channels=3),
                INPUT_SHAPE[:2]
            ), tf.float32)
        ), l
    ), num_parallel_calls=AUTOTUNE)
    .batch(SPEED_BATCH)
    .prefetch(AUTOTUNE)
)

# Warm-up: 1 batch so XLA/cuDNN doesn't count compile time
warmup_batch = next(iter(speed_ds.take(1)))[0]
_ = model.predict(warmup_batch, verbose=0)

total_images = 0
total_time   = 0.0

for i, (batch_x, _) in enumerate(speed_ds):
    if i >= MAX_BATCHES:
        break
    n = len(batch_x)
    t0 = time.perf_counter()
    model.predict(batch_x, verbose=0)
    t1 = time.perf_counter()
    total_images += n
    total_time   += (t1 - t0)

avg_ms  = (total_time / total_images) * 1000
fps     = total_images / total_time

print("\n" + "="*60)
print("Inference Speed:")
print(f"  FPS          : {fps:.2f}")
print(f"  ms / image   : {avg_ms:.2f}")
print(f"  Batch size   : {SPEED_BATCH}")
print(f"  Batches used : {min(i+1, MAX_BATCHES)}")
print(f"  Total images : {total_images}")
print(f"  Total time   : {total_time:.3f} s")
print("="*60)

with open(os.path.join(RESULT_DIR, "inference_speed.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("Inference Speed:\n")
    f.write(f"  FPS        : {fps:.2f}\n")
    f.write(f"  ms/image   : {avg_ms:.2f}\n")
    f.write(f"  Batch size : {SPEED_BATCH}\n")
    f.write(f"  Total imgs : {total_images}\n")
    f.write(f"  Total time : {total_time:.3f}s\n")
    f.write("="*60 + "\n")

print("Inference speed saved.")


In [ ]:
# ========== TOP-K ACCURACY ========== #
from sklearn.metrics import top_k_accuracy_score

n_classes = len(class_names)

# Guard against k > number of classes.
top_1_acc = top_k_accuracy_score(y_true, pred_probs, k=1, labels=range(n_classes))
top_3_k = min(3, n_classes)
top_5_k = min(5, n_classes)
top_3_acc = top_k_accuracy_score(y_true, pred_probs, k=top_3_k, labels=range(n_classes))
top_5_acc = top_k_accuracy_score(y_true, pred_probs, k=top_5_k, labels=range(n_classes))

print("\n" + "="*60)
print("Top-K Accuracy:")
print(f"  Top-1 Accuracy: {top_1_acc:.4f} ({top_1_acc*100:.2f}%)")
print(f"  Top-{top_3_k} Accuracy: {top_3_acc:.4f} ({top_3_acc*100:.2f}%)")
print(f"  Top-{top_5_k} Accuracy: {top_5_acc:.4f} ({top_5_acc*100:.2f}%)")
print("="*60)

with open(os.path.join(RESULT_DIR, "topk_accuracy.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("Top-K Accuracy:\n")
    f.write(f"  Top-1 Accuracy: {top_1_acc:.4f} ({top_1_acc*100:.2f}%)\n")
    f.write(f"  Top-{top_3_k} Accuracy: {top_3_acc:.4f} ({top_3_acc*100:.2f}%)\n")
    f.write(f"  Top-{top_5_k} Accuracy: {top_5_acc:.4f} ({top_5_acc*100:.2f}%)\n")
    f.write("="*60 + "\n")

print("Top-K accuracy saved")


In [ ]:
# ========== PER-CLASS ACCURACY ========== #
from sklearn.metrics import classification_report

# Get per-class metrics
report_dict = classification_report(y_true, y_pred, target_names=class_names, 
                                   output_dict=True, digits=4)

# Create per-class accuracy dataframe
per_class_df = pd.DataFrame({
    'Class': class_names,
    'Precision': [report_dict[c]['precision'] for c in class_names],
    'Recall': [report_dict[c]['recall'] for c in class_names],
    'F1-Score': [report_dict[c]['f1-score'] for c in class_names],
    'Support': [report_dict[c]['support'] for c in class_names]
})

print("\n" + "="*60)
print("Per-Class Metrics:")
print(per_class_df.to_string(index=False))
print("="*60)

# Save to CSV
per_class_df.to_csv(os.path.join(RESULT_DIR, "per_class_metrics.csv"), index=False)
print("✅ Per-class metrics saved")


In [ ]:
# ========== PREDICTIONS CSV ========== #
# Create detailed predictions dataframe
predictions_df = pd.DataFrame({
    'filename': test_filenames,
    'true_label': [class_names[i] for i in y_true],
    'predicted_label': [class_names[i] for i in y_pred],
    'correct': y_true == y_pred,
    'confidence': [pred_probs[i][y_pred[i]] for i in range(len(y_pred))]
})

# Add top-3 predictions for each image
for k in range(min(3, len(class_names))):
    top_k_indices = np.argsort(pred_probs, axis=1)[:, -(k+1)]
    predictions_df[f'top_{k+1}_class'] = [class_names[i] for i in top_k_indices]
    predictions_df[f'top_{k+1}_prob'] = [pred_probs[i][top_k_indices[i]] for i in range(len(pred_probs))]

# Save to CSV
predictions_df.to_csv(os.path.join(RESULT_DIR, "predictions_detail.csv"), index=False)

print(f"✅ Predictions CSV saved ({len(predictions_df)} samples)")
print(f"   Correct predictions: {predictions_df['correct'].sum()}")
print(f"   Incorrect predictions: {(~predictions_df['correct']).sum()}")


In [ ]:
# ========== COMPREHENSIVE SUMMARY REPORT ========== #
summary_report = []
summary_report.append("="*80)
summary_report.append("V1 GAP+GMP - COMPREHENSIVE EVALUATION REPORT")
summary_report.append("="*80)
summary_report.append(f"\nDataset: {DATA_DIR}")
summary_report.append(f"Result Directory: {RESULT_DIR}")
summary_report.append(f"Training Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")

summary_report.append("\n" + "-"*80)
summary_report.append("MODEL CONFIGURATION")
summary_report.append("-"*80)
summary_report.append("Architecture: EfficientNetB4 GAP+GMP classifier")
summary_report.append(f"Input Shape: {INPUT_SHAPE}")
summary_report.append(f"Number of Classes: {len(class_names)}")
summary_report.append(f"Classes: {', '.join(plain_class_names(class_names))}")
summary_report.append(f"\nTotal Parameters: {total_params:,}")
summary_report.append(f"Trainable Parameters: {trainable_params:,}")
summary_report.append(f"Non-trainable Parameters: {non_trainable_params:,}")
summary_report.append(f"Model Size: {model_size_mb:.2f} MB")

summary_report.append("\n" + "-"*80)
summary_report.append("DATASET STATISTICS")
summary_report.append("-"*80)
summary_report.append(f"Training Samples:   {n_train}")
summary_report.append(f"Validation Samples: {n_val}")
summary_report.append(f"Test Samples:       {n_test}")

summary_report.append("\n" + "-"*80)
summary_report.append("TRAINING CONFIGURATION")
summary_report.append("-"*80)
summary_report.append(f"Batch Size: {BATCH_SIZE}")
summary_report.append(f"Total Epochs (actual): {len(history.history['loss'])}")
summary_report.append(f"Initial Learning Rate: {LR}")
summary_report.append("Optimizer: AdamW + CosineDecay")
summary_report.append(f"Loss Function: {LOSS_DESCRIPTION}")
summary_report.append(f"Label Smoothing: {LABEL_SMOOTHING}")
summary_report.append(f"TTA Rounds: {TTA_ROUNDS}")

summary_report.append("\n" + "-"*80)
summary_report.append("PERFORMANCE METRICS")
summary_report.append("-"*80)
summary_report.append(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
summary_report.append(f"Top-1 Accuracy: {top_1_acc:.4f} ({top_1_acc*100:.2f}%)")
summary_report.append(f"Top-{min(3, len(class_names))} Accuracy: {top_3_acc:.4f} ({top_3_acc*100:.2f}%)")
summary_report.append(f"Top-{min(5, len(class_names))} Accuracy: {top_5_acc:.4f} ({top_5_acc*100:.2f}%)")

summary_report.append("\n" + "-"*80)
summary_report.append("INFERENCE SPEED")
summary_report.append("-"*80)
summary_report.append(f"FPS: {fps:.2f}")
summary_report.append(f"ms/image: {avg_ms:.2f}")

summary_report.append("\n" + "-"*80)
summary_report.append("BEST TRAINING EPOCH METRICS")
summary_report.append("-"*80)
best_val_loss_idx = np.argmin(history.history['val_loss'])
summary_report.append(f"Best Epoch: {best_val_loss_idx + 1}")
summary_report.append(f"  Train Loss:     {history.history['loss'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Train Accuracy: {history.history['accuracy'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Val Loss:       {history.history['val_loss'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Val Accuracy:   {history.history['val_accuracy'][best_val_loss_idx]:.4f}")

summary_report.append("\n" + "="*80)
summary_report.append("END OF REPORT")
summary_report.append("="*80)

summary_text = "\n".join(summary_report)
print(summary_text)

with open(os.path.join(RESULT_DIR, "SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nComprehensive summary report saved.")


In [ ]:
# ========== LIST ALL REPORT FILES ========== #
import glob

print("\n" + "="*80)
print("GENERATED REPORT FILES:")
print("="*80)

all_files = glob.glob(os.path.join(RESULT_DIR, "*"))
for file_path in sorted(all_files):
    if os.path.isfile(file_path):
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path)
        if file_size < 1024:
            size_str = f"{file_size} B"
        elif file_size < 1024*1024:
            size_str = f"{file_size/1024:.2f} KB"
        else:
            size_str = f"{file_size/(1024*1024):.2f} MB"
        print(f"  ✓ {file_name:40s} ({size_str})")
    elif os.path.isdir(file_path):
        dir_name = os.path.basename(file_path)
        num_files = len([f for f in glob.glob(os.path.join(file_path, "**/*"), recursive=True) if os.path.isfile(f)])
        print(f"  📁 {dir_name:40s} ({num_files} files)")

print("="*80)


In [ ]:
# ========== ZIP ALL REPORTS ========== #
import shutil

zip_output_path = f"/kaggle/working/{STRATEGY_KEY}_SingleRun_Report"
print("Creating complete report archive...")
print(f"Source: {RESULT_DIR}")
print(f"Output: {zip_output_path}.zip")

shutil.make_archive(zip_output_path, 'zip', RESULT_DIR)

zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print("Complete report archived successfully!")
print(f"Archive size: {zip_size:.2f} MB")
print(f"Location: {zip_output_path}.zip")
print("" + "="*80)
print("ARCHIVE CONTENTS:")
print("  - SUMMARY_REPORT.txt")
print("  - model_summary.txt")
print("  - model_size.txt")
print("  - inference_speed.txt")
print("  - topk_accuracy.txt")
print("  - classification_report.txt")
print("  - per_class_metrics.csv")
print("  - predictions_detail.csv")
print("  - training_log.csv")
print("  - learning_curves.png")
print("  - confusion_matrix.png")
print("  - best_model.keras")
print("  - qualitative_analysis/")
print("="*80)
